# Revision Experiments for the Dyslexia EEG Pipeline Paper

This notebook runs every experiment requested in the critical review. Run it **in the same folder as
`dyslexia_pipeline_adaptive.ipynb`**, so that `eeg_adaptive.py` is available. If the module file is
missing, it is extracted automatically from the original notebook.

| Exp. | What it does | Review item |
|---|---|---|
| **E0** | Simulator v2: 1/f exponent, alpha peak and band amplitudes vary between subjects; age effects; a per-subject effect size; volume conduction from 19 cortical sources; blink and EMG artefacts | 4A, Limitations |
| **E1** | Effect-size sweep, ε = 0 to 0.30, with ε = 0 as a null check; v1 (original) and v2 simulators; repeated over seeds | Top-5 #1, 5C |
| **E2** | Whole pipeline (generation included) repeated over 10 seeds; hold-out and subject-level CV; sensitivity at 90 % specificity; a decision threshold tuned inside the training folds | 4D, Top-5 #5 |
| **E3** | Leave-one-montage-out, leave-one-sampling-rate-out, and an unseen EGI-128 cohort | Top-5 #2, 5A |
| **E4** | Component ablation and naive baselines; spherical-spline interpolation; missingness indicators; HistGradientBoosting; midline split; P7/P8 treated as temporal | Top-5 #3, 4B, 4C |
| **E5** | Confound and leakage tests: can montage, sampling rate, unit, gain or age be predicted from the features? Correlation of P(dyslexic) with nuisance variables within each class | 4C, 5C (Fig. 4), 5E |
| **E6** | Calibration (reliability diagram, Brier score, ECE) and the confidence score | 5B |
| **E7** | Electrode-placement study: accuracy against channel count and regions covered; drop-one-region; named layouts | 5C, Future work |
| **E8** | Classifier comparison with valid tests: paired across seeds, Nadeau–Bengio corrected t-test, DeLong test | Abstract, 5D |
| **E9** | Permutation importance on held-out data, with readable labels and feature-family groups | 4C, Fig. 5 |
| **E10** | Auto-detection counts (detected / affected, flagged / unaffected) | 5A |
| **E11** | Example simulated spectra, by class and montage (v1 and v2) | Figures |
| **E12** | Sensitivity of the EGI 0.38 site-distance threshold | 4B |
| **E13** | Accuracy against recording length | 4A |

**Outputs.** Everything is written to `revision_results/`: CSV tables, figures in PNG and PDF, and
`results_summary.txt`. A zip of the folder is made at the end. **Send me `results_summary.txt`** (and
the zip if you can), and I will revise the paper from it.

**Runtime.** On an 8-core machine the full run takes about 2 to 4 hours; with 4 cores, leave it overnight.
Set `QUICK = True` first for a 10 to 20 minute smoke test. Requires `mne` (already used by the pipeline for EGI). Each experiment can be switched off in `RUN`. Generated cohorts are cached in
`revision_results/cache/`, so a rerun after a crash skips the work already done.

## 0. Configuration

In [ ]:
import hashlib
import importlib
import json
import os
import pickle
import platform
import re
import sys
import time
import warnings
import zipfile
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy import signal, stats
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (GradientBoostingClassifier, HistGradientBoostingClassifier,
                              RandomForestClassifier, RandomForestRegressor)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, brier_score_loss, r2_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold, KFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore", category=RuntimeWarning)       # nanmean of empty slices etc.
warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------------------------------------------------------- run size
QUICK = False                                # True = small smoke test (~10-20 min) before the full run
if os.environ.get("REVISION_QUICK") == "1":
    QUICK = True

N_SUBJECTS = 120 if QUICK else 360           # per cohort, balanced classes (as in the paper)
SEEDS_MAIN = list(range(2 if QUICK else 10)) # E2/E5/E6/E8: whole pipeline repeated (review: >= 10)
SEEDS_SWEEP = list(range(2 if QUICK else 5)) # E1/E3/E4/E9
SEEDS_SMALL = list(range(1 if QUICK else 3)) # E7/E12/E13
EFFECT_GRID = [0.0, 0.3, 0.6] if QUICK else [0.0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.60, 0.80]
ORIGINAL_EFFECT = 0.30                       # the value used in the submitted paper
MAIN_EFFECT = 0.60                           # main operating point of the revision (v2): Cohen d ~0.8 on global log TBR
N_TEST_E7 = 40 if QUICK else 120             # test subjects for the electrode-placement study
N_EGI = 30 if QUICK else 80                  # subjects in each unseen EGI-128 cohort
K_FOLDS = 5
N_JOBS = -1                                  # all cores for cohort generation
N_JOBS_MODEL = -1                            # cores for the random forest

RUN = dict(E1=True, E2=True, E3=True, E4=True, E5=True, E6=True, E7=True,
           E8=True, E9=True, E10=True, E11=True, E12=True, E13=True)

FS_CHOICES = [128, 250, 256, 500, 512, 1000]
DURATION_CHOICES_S = [24, 32, 40]
MONTAGE_P = {"full19": 0.35, "openbci8": 0.25, "random": 0.40}

WORK_DIR = Path.cwd()
RES = WORK_DIR / "revision_results"
FIG = RES / "figures"
CACHE = RES / "cache"
for d in (RES, FIG, CACHE):
    d.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------- the pipeline module
MODULE_PATH = WORK_DIR / "eeg_adaptive.py"
if not MODULE_PATH.exists():
    src_nb = WORK_DIR / "dyslexia_pipeline_adaptive.ipynb"
    if not src_nb.exists():
        raise FileNotFoundError("eeg_adaptive.py not found. Run this notebook in the folder of "
                                "dyslexia_pipeline_adaptive.ipynb (or run that notebook once first).")
    cells = json.loads(src_nb.read_text(encoding="utf-8"))["cells"]
    code = next("".join(c["source"]) for c in cells if "MODULE_SRC = r'''" in "".join(c["source"]))
    MODULE_PATH.write_text(code.split("r'''", 1)[1].rsplit("'''", 1)[0], encoding="utf-8")
    print("extracted eeg_adaptive.py from the original notebook")

os.environ["PYTHONPATH"] = str(WORK_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")  # for workers
sys.path.insert(0, str(WORK_DIR))
import eeg_adaptive as eeg
importlib.reload(eeg)

try:
    import mne
    mne.set_log_level("ERROR")
    HAVE_MNE = True
except ImportError:
    HAVE_MNE = False

import sklearn, scipy
try:
    import psutil
    RAM_GB = psutil.virtual_memory().total / 2 ** 30
except ImportError:
    RAM_GB = float("nan")
HARDWARE = (f"{platform.system()} {platform.release()} | CPU: {platform.processor() or platform.machine()} "
            f"| {os.cpu_count()} logical cores | RAM {RAM_GB:.0f} GB | Python {platform.python_version()} "
            f"| numpy {np.__version__} | scipy {scipy.__version__} | scikit-learn {sklearn.__version__} "
            f"| MNE {mne.__version__ if HAVE_MNE else 'not installed'}")
print(HARDWARE)
print(f"QUICK={QUICK} | subjects/cohort={N_SUBJECTS} | seeds main={len(SEEDS_MAIN)} sweep={len(SEEDS_SWEEP)}")
if not HAVE_MNE:
    print("WARNING: MNE not installed -> E3 (EGI cohort), E12 and the spline-interpolation ablation are skipped. "
          "pip install mne")

SUMMARY = []          # lines collected for results_summary.txt


def log(title, obj=None):
    """Print and store a result block for results_summary.txt."""
    block = f"\n=== {title} ===\n" + ("" if obj is None else (obj.to_string() if hasattr(obj, "to_string") else str(obj)))
    print(block)
    SUMMARY.append(block)


def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=300, bbox_inches="tight")


plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "savefig.bbox": "tight", "figure.dpi": 110})
C_CTRL, C_DYS, C_ACC = "#4C72B0", "#C44E52", "#55A868"

## 1. Simulators and configurable pipeline (`revision_sim.py`)
The next cell writes `revision_sim.py`, which is the code to release with the paper (review item 5). It contains:
* **Geometry and volume conduction.** There are 19 cortical sources, one under each 10-20 site. Each sensor
  records a Gaussian-weighted mixture of the sources (kernel width `vc_lambda` on the unit sphere), so
  neighbouring channels are correlated and the average reference behaves realistically. EGI-128 sensors use
  their own 3-D positions.
* **`simulate_v1`** is the paper's original simulator, unchanged apart from the effect size being a parameter.
* **`simulate_v2`** is the realistic simulator. It varies the 1/f exponent, alpha peak frequency (which
  rises with age), band amplitudes and age between subjects; draws a per-subject effect
  ε_i ~ ε·(1 + 0.5·N(0,1)); generates narrow-band rhythms instead of fixed sinusoids; applies volume
  conduction; and adds blink and temporal-EMG artefacts. All of these are independent of class.
* **`prep()`**, a copy of `eeg.preprocess_recording` in which each step can be switched off for the
  ablations, plus feature variants (relative or absolute power, region- or channel-level, midline split,
  P7/P8 treated as temporal, spherical-spline interpolation).

**Class effect (both simulators).** The theta amplitude is multiplied by (1 + ε_i·w), and the alpha and beta
amplitudes by (1 − 0.5·ε_i·w), with w = 1.6 over left temporal, parietal and occipital sources.
For ε = 0.30 this is +69 % theta power and −28 % alpha/beta power in the affected regions.

In [ ]:
%%writefile revision_sim.py
"""revision_sim.py - simulators (v1 original, v2 realistic) and a configurable copy of the
eeg_adaptive pipeline, used by the revision experiments. Released with the paper for reproducibility.

Every function is deterministic given a numpy Generator; cohort workers derive one seed per subject.
"""
import re
from functools import lru_cache

import numpy as np
from scipy import signal

import eeg_adaptive as eeg

try:
    import mne
    mne.set_log_level("ERROR")
    HAVE_MNE = True
except ImportError:
    HAVE_MNE = False

BASE_CH = ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T7", "C3", "Cz",
           "C4", "T8", "P7", "P3", "Pz", "P4", "P8", "O1", "O2"]
BASE_CANON = [eeg.normalize_channel_name(c) for c in BASE_CH]          # 'FP1', 'FZ', ...
OPENBCI8 = ["Fp1", "Fp2", "C3", "C4", "P7", "P8", "O1", "O2"]
OPENBCI8_CANON = [eeg.normalize_channel_name(c) for c in OPENBCI8]
EGI_NAMES = [f"E{i}" for i in range(1, 129)]

# analytic fallback (polar angle from Cz, azimuth from the nose, degrees) if MNE is not installed
_ANALYTIC = {"FP1": (72, -18), "FP2": (72, 18), "F7": (72, -54), "F3": (49, -40), "FZ": (36, 0),
             "F4": (49, 40), "F8": (72, 54), "T7": (72, -90), "C3": (36, -90), "CZ": (0, 0),
             "C4": (36, 90), "T8": (72, 90), "P7": (72, -126), "P3": (49, -140), "PZ": (36, 180),
             "P4": (49, 140), "P8": (72, 126), "O1": (72, -162), "O2": (72, 162), "FPZ": (72, 0)}


def _sph(theta, phi):
    t, p = np.radians(theta), np.radians(phi)
    return np.array([np.sin(t) * np.sin(p), np.sin(t) * np.cos(p), np.cos(t)])


@lru_cache(maxsize=None)
def _montage_xyz(montage_name):
    pos = mne.channels.make_standard_montage(montage_name).get_positions()["ch_pos"]
    names = list(pos)
    xyz = eeg._unit_sphere([pos[n] for n in names])
    return {n.upper(): xyz[i] for i, n in enumerate(names)}


def positions(names, net="1020"):
    """Unit-sphere xyz (x right, y front, z up) for canonical 10-20 names or EGI E-names."""
    if net == "egi":
        d = _montage_xyz("GSN-HydroCel-128")
        return np.array([d[n.upper()] for n in names])
    if HAVE_MNE:
        d = _montage_xyz("standard_1005")
        return np.array([d[n] for n in names])
    return np.array([_sph(*_ANALYTIC[n]) for n in names])


def mixing_matrix(sensor_xyz, source_xyz, lam):
    d2 = ((sensor_xyz[:, None, :] - source_xyz[None, :, :]) ** 2).sum(-1)
    m = np.exp(-d2 / (2 * lam ** 2))
    return m / m.sum(axis=1, keepdims=True)


def n_regions(canon_names):
    return len({eeg.channel_region(c) for c in canon_names})


SRC_XYZ = positions(BASE_CANON)


SIM2 = dict(
    bg_rms_uv=10.0,                 # 1/f background per source
    chi_range=(1.0, 2.0),           # 1/f power exponent per subject (amplitude ~ f^-chi/2)
    rhythms={                       # base amplitude (sinusoid-equivalent peak, uV), bandwidth (Hz), effect sign
        "theta": dict(base_uv=2.5, bw=1.0, sign=+1.0, f_mean=6.0, f_sd=0.5, f_clip=(4.5, 7.5)),
        "alpha": dict(base_uv=4.0, bw=1.0, sign=-0.5),          # peak frequency depends on age (below)
        "beta": dict(base_uv=1.5, bw=2.5, sign=-0.5, f_mean=20.0, f_sd=2.5, f_clip=(15.0, 26.0)),
    },
    topography={                    # amplitude multiplier per lobe (alpha posterior, theta fronto-midline)
        "theta": {"F": 1.2, "C": 1.0, "T": 1.0, "P": 1.0, "O": 1.0, "M": 1.2},
        "alpha": {"F": 0.6, "C": 1.0, "T": 1.0, "P": 1.4, "O": 1.6, "M": 1.0},
        "beta": {"F": 1.0, "C": 1.2, "T": 1.0, "P": 1.0, "O": 1.0, "M": 1.1},
    },
    subj_amp_sd=0.30,               # lognormal SD of each rhythm's amplitude between subjects
    source_amp_sd=0.15,             # lognormal SD between sources within a subject
    source_freq_sd=0.2,             # Hz jitter of the peak between sources
    effect_sd_frac=0.5,             # eps_i = eps * (1 + 0.5 N(0,1)), drawn per dyslexic subject
    effect_regions=("T_L", "P_L", "O_L"), effect_region_gain=1.6,
    age_range=(7.0, 12.0),          # years, uniform, independent of class
    alpha_pf=dict(intercept=8.0, slope=0.30, sd=0.6, clip=(7.0, 11.5)),   # Hz at age 7, Hz/year
    theta_age_slope=-0.05,          # theta amplitude falls 5 %/year (developmental, class-independent)
    vc_lambda=0.45,                 # volume-conduction kernel width (unit-sphere chord)
    sensor_noise_uv=3.0,            # independent 1/f sensor noise
    scale_to_uv=15.0,               # median sensor SD after mixing, before gain
    gain_sd=0.25,                   # amplifier/skin gain, lognormal, multiplies everything
    blink=dict(p=0.5, rate_hz=(0.1, 0.4), amp_uv=(80.0, 200.0), dur_s=(0.25, 0.40), width=0.40),
    emg=dict(p=0.3, rms_uv=(5.0, 15.0), bursts=(2, 6), burst_s=(1.0, 4.0), width=0.35, hp_hz=20.0),
    transient_p=0.3, mains_p=0.7, bad_p=0.15,
)

_SRC_LOBE = [("M" if eeg.channel_region(c) == "MID" else eeg.channel_region(c)[0]) for c in BASE_CANON]
_SRC_REGION = [eeg.channel_region(c) for c in BASE_CANON]


def _colored(n, fs, amp, rng, k):
    """k unit-variance Gaussian processes with amplitude spectrum amp (len(f)) or (k, len(f))."""
    f = np.fft.rfftfreq(n, 1 / fs)
    a = amp(f) if callable(amp) else amp
    spec = (rng.standard_normal((k, len(f))) + 1j * rng.standard_normal((k, len(f)))) * a
    spec[..., 0] = 0
    x = np.fft.irfft(spec, n=n, axis=-1)
    return x / (x.std(axis=-1, keepdims=True) + 1e-12)


def _add_nuisance(sig, t, fs, rng, mains, bad, unit, ch_count, mains_p, bad_p, transient_p):
    """DC + drift, mains, movement transient, bad channel, units (as in the original v1 simulator)."""
    n = sig.shape[1]
    sig += rng.normal(0, 50, (ch_count, 1)) + 30 * np.sin(2 * np.pi * 0.1 * t + rng.uniform(0, 6.28))
    if mains == "random":
        mains = float(rng.choice([50.0, 60.0])) if rng.random() < mains_p else None
    if mains is not None and mains < fs / 2 - 1:
        sig += rng.uniform(2, 8) * np.sin(2 * np.pi * mains * t + rng.uniform(0, 6.28))
    else:
        mains = None
    transient = rng.random() < transient_p
    if transient:
        c = int(rng.integers(int(0.5 * fs), n - int(0.5 * fs)))
        half = int(0.15 * fs)
        sig[:, c - half:c + half] += 500 * np.hanning(2 * half)
    if bad == "random":
        bad = str(rng.choice(["flat", "noisy"])) if rng.random() < bad_p else None
    bad_idx = None
    if bad is not None:
        bad_idx = int(rng.integers(ch_count))
        sig[bad_idx] = rng.normal(0, 300, n) if bad == "noisy" else 0.0
    unit = unit or str(rng.choice(["V", "uV"]))
    if unit == "V":
        sig = sig * 1e-6
    return sig, dict(unit=unit, mains=mains, bad_type=bad, bad_idx=bad_idx, transient=bool(transient))


def simulate_v1(label, canon, fs, duration_s, rng, effect, unit=None, mains="random", bad="random"):
    """The paper's original simulator (unchanged apart from the effect parameter)."""
    n, nch = int(fs * duration_s), len(canon)
    freqs = np.fft.rfftfreq(n, 1 / fs)
    shaping = np.ones_like(freqs)
    shaping[1:] = 1 / np.sqrt(freqs[1:])
    sig = np.fft.irfft(np.fft.rfft(rng.standard_normal((nch, n)), axis=-1) * shaping, n=n, axis=-1)
    sig = 10 * sig / sig.std(axis=-1, keepdims=True)
    t = np.arange(n) / fs
    w = np.array([1.6 if eeg.channel_region(c) in ("T_L", "P_L", "O_L") else 1.0 for c in canon])[:, None]
    gain = rng.lognormal(0, 0.25)
    for centre, base, eff in [(6.0, 2.5, +effect), (10.0, 4.0, -0.5 * effect), (20.0, 1.5, -0.5 * effect)]:
        phase = rng.uniform(0, 2 * np.pi, (nch, 1))
        sig += base * gain * (1 + eff * w * label) * np.sin(2 * np.pi * (centre + rng.normal(0, 0.5)) * t + phase)
    sig, info = _add_nuisance(sig, t, fs, rng, mains, bad, unit, nch, 0.7, 0.15, 0.3)
    info.update(gain=gain, eps_i=effect if label else 0.0, age=np.nan, alpha_pf=10.0, chi=1.0,
                blink=False, emg=False)
    return sig, info


def simulate_v2(label, sensor_names, fs, duration_s, rng, effect, net="1020",
                unit=None, mains="random", bad="random", p=SIM2):
    """Realistic simulator: subject biology + volume conduction + physiological artefacts."""
    n, nch = int(fs * duration_s), len(sensor_names)
    t = np.arange(n) / fs
    f = np.fft.rfftfreq(n, 1 / fs)
    # -- subject-level biology (independent of class)
    age = rng.uniform(*p["age_range"])
    chi = rng.uniform(*p["chi_range"])
    ap = p["alpha_pf"]
    pf = {"alpha": float(np.clip(ap["intercept"] + ap["slope"] * (age - 7) + rng.normal(0, ap["sd"]), *ap["clip"]))}
    for r in ("theta", "beta"):
        q = p["rhythms"][r]
        pf[r] = float(np.clip(rng.normal(q["f_mean"], q["f_sd"]), *q["f_clip"]))
    amp = {r: rng.lognormal(0, p["subj_amp_sd"]) for r in p["rhythms"]}
    amp["theta"] *= max(0.2, 1 + p["theta_age_slope"] * (age - 7))
    eps_i = effect * (1 + p["effect_sd_frac"] * rng.standard_normal()) if (label and effect > 0) else 0.0
    # -- 19 cortical sources
    bg_amp = np.where(f > 0, np.maximum(f, 0.5) ** (-chi / 2), 0.0)
    S = p["bg_rms_uv"] * _colored(n, fs, bg_amp, rng, 19)
    w_eff = np.array([p["effect_region_gain"] if r in p["effect_regions"] else 1.0 for r in _SRC_REGION])
    for r, q in p["rhythms"].items():
        fc = pf[r] + rng.normal(0, p["source_freq_sd"], 19)
        shape = np.exp(-(f[None, :] - fc[:, None]) ** 2 / (2 * q["bw"] ** 2))
        x = _colored(n, fs, shape, rng, 19)
        topo = np.array([p["topography"][r][lb] for lb in _SRC_LOBE])
        jitter = rng.lognormal(0, p["source_amp_sd"], 19)
        cls = np.clip(1 + q["sign"] * eps_i * w_eff * label, 0.1, None)
        S += (q["base_uv"] / np.sqrt(2) * amp[r] * topo * jitter * cls)[:, None] * x
    # -- volume conduction to the sensors
    sens_xyz = positions(sensor_names, net)
    X = mixing_matrix(sens_xyz, SRC_XYZ, p["vc_lambda"]) @ S
    X += p["sensor_noise_uv"] * _colored(n, fs, bg_amp, rng, nch)
    X *= p["scale_to_uv"] / np.median(X.std(axis=1))
    gain = rng.lognormal(0, p["gain_sd"])
    X *= gain
    # -- physiological artefacts (class-independent)
    b = p["blink"]
    blink = rng.random() < b["p"]
    if blink:
        w = np.exp(-((sens_xyz - _sph(100, 0)) ** 2).sum(1) / (2 * b["width"] ** 2))[:, None]
        rate = rng.uniform(*b["rate_hz"])
        for tc in np.cumsum(rng.exponential(1 / rate, int(rate * duration_s * 2) + 2)):
            d = rng.uniform(*b["dur_s"])
            i0, i1 = int(tc * fs), int((tc + d) * fs)
            if i1 >= n:
                break
            X[:, i0:i1] += w * rng.uniform(*b["amp_uv"]) * gain * np.sin(np.linspace(0, np.pi, i1 - i0))
    e = p["emg"]
    emg = rng.random() < e["p"]
    if emg:
        sides = {"L": [_sph(*_ANALYTIC["T7"])], "R": [_sph(*_ANALYTIC["T8"])]}
        centres = sides[str(rng.choice(["L", "R"]))] if rng.random() < 0.5 else sides["L"] + sides["R"]
        w = np.max([np.exp(-((sens_xyz - c) ** 2).sum(1) / (2 * e["width"] ** 2)) for c in centres], axis=0)[:, None]
        sos = signal.butter(4, e["hp_hz"], btype="highpass", fs=fs, output="sos")
        noise = signal.sosfilt(sos, rng.standard_normal((nch, n)), axis=-1)
        noise /= noise.std(axis=-1, keepdims=True)
        env = np.zeros(n)
        for _ in range(int(rng.integers(e["bursts"][0], e["bursts"][1] + 1))):
            L = int(rng.uniform(*e["burst_s"]) * fs)
            s0 = int(rng.integers(0, max(1, n - L)))
            env[s0:s0 + L] = np.maximum(env[s0:s0 + L], signal.windows.tukey(len(env[s0:s0 + L]), 0.3))
        X += w * rng.uniform(*e["rms_uv"]) * gain * noise * env
    X, info = _add_nuisance(X, t, fs, rng, mains, bad, unit, nch, p["mains_p"], p["bad_p"], p["transient_p"])
    info.update(gain=gain, eps_i=eps_i, age=age, alpha_pf=pf["alpha"], chi=chi, blink=bool(blink), emg=bool(emg))
    return X, info


def style_names(canon_plain, rng, style=None):
    """Channel-name styles from the paper ('Fp1', 'EEG Fp1-REF', 'FP1', legacy T3/T5)."""
    style = style or str(rng.choice(["plain", "ref", "upper", "legacy"], p=[0.4, 0.2, 0.2, 0.2]))
    pretty = {c: n for c, n in zip(BASE_CANON, BASE_CH)}
    legacy = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}
    fmt = {"plain": lambda n: n, "ref": lambda n: f"EEG {n}-REF", "upper": lambda n: n.upper(),
           "legacy": lambda n: legacy.get(n, n)}[style]
    return [fmt(pretty[c]) for c in canon_plain], style


MONTAGE_P = {"full19": 0.35, "openbci8": 0.25, "random": 0.40}     # as in the paper


def random_montage(rng, kind=None, probs=None):
    probs = probs or MONTAGE_P
    kind = kind or str(rng.choice(list(probs), p=list(probs.values())))
    if kind == "full19":
        return kind, list(BASE_CANON)
    if kind == "openbci8":
        return kind, list(OPENBCI8_CANON)
    while True:
        idx = sorted(rng.choice(19, int(rng.integers(8, 19)), replace=False))
        ch = [BASE_CANON[i] for i in idx]
        if n_regions(ch) >= 4:
            return kind, ch


PREP_CFGS = {
    "full":      dict(names=True, unit=True, notch=True, bad=True, car=True, reject=True),
    "no_names":  dict(names=False, unit=True, notch=True, bad=True, car=True, reject=True),
    "no_unit":   dict(names=True, unit=False, notch=True, bad=True, car=True, reject=True),
    "no_notch":  dict(names=True, unit=True, notch=False, bad=True, car=True, reject=True),
    "no_bad":    dict(names=True, unit=True, notch=True, bad=False, car=True, reject=True),
    "no_car":    dict(names=True, unit=True, notch=True, bad=True, car=False, reject=True),
    "no_reject": dict(names=True, unit=True, notch=True, bad=True, car=True, reject=False),
    "naive":     dict(names=False, unit=False, notch=False, bad=False, car=False, reject=False),
}
_EXACT = set(BASE_CH)          # a conventional pipeline expects exactly these labels
DEFAULT_SITE_DISTANCE = eeg._MAX_SITE_DISTANCE


def prep(data, fs, names, cfg, site_dist=None):
    """Raw recording -> (epochs, canonical names, report). Raises ValueError if unusable."""
    rep = {}
    data = np.asarray(data, dtype=float)
    if cfg["names"]:
        eeg._MAX_SITE_DISTANCE = DEFAULT_SITE_DISTANCE if site_dist is None else site_dist
        keep, canon, dropped, rep["mapping"] = eeg.resolve_channels(list(names), None, 4)
    else:
        keep = [i for i, nm in enumerate(names) if nm in _EXACT]
        canon = [eeg.normalize_channel_name(names[i]) for i in keep]
        if n_regions(canon) < 4:
            raise ValueError("exact-name matching found < 4 regions")
    rep["n_mapped"] = len(keep)
    data = data[keep]
    if cfg["unit"]:
        data, rep["units"] = eeg.to_microvolts(data, "auto")
    data = data - data.mean(axis=1, keepdims=True)
    rep["mains_hz"] = eeg.detect_mains(data, fs) if cfg["notch"] else None
    if rep["mains_hz"]:
        b, a = signal.iirnotch(rep["mains_hz"], Q=30.0, fs=fs)
        data = signal.filtfilt(b, a, data, axis=-1)
    data = eeg.resample_to(data, fs)
    sos = signal.butter(4, eeg.BANDPASS, btype="bandpass", fs=eeg.TARGET_FS, output="sos")
    data = signal.sosfiltfilt(sos, data, axis=-1)
    rep["bad_idx"] = []
    if cfg["bad"]:
        bad = eeg.find_bad_channels(data)
        rep["bad_idx"] = [keep[i] for i in np.flatnonzero(bad)]
        data, canon = data[~bad], [c for c, b_ in zip(canon, bad) if not b_]
    if n_regions(canon) < 4:
        raise ValueError("< 4 regions after bad-channel removal")
    if cfg["car"] and len(canon) >= 4:
        data = data - data.mean(axis=0, keepdims=True)
    epochs = eeg.make_epochs(data)
    if cfg["reject"]:
        ptp = np.ptp(epochs, axis=-1).max(axis=1)
        if len(ptp) >= 3:
            epochs = epochs[ptp <= 3.0 * np.median(ptp)]
    rep["n_epochs"] = len(epochs)
    rep["regions"] = n_regions(canon)
    return epochs, canon, rep


# ----------------------------------------------------------------------------- feature variants
def band_powers(epochs):
    freqs, psd = signal.welch(epochs, fs=eeg.TARGET_FS, nperseg=int(2 * eeg.TARGET_FS), axis=-1)
    df = freqs[1] - freqs[0]

    def band(lo, hi):
        m = (freqs >= lo) & (freqs < hi)
        return psd[..., m].sum(axis=-1) * df
    bp = np.stack([band(lo, hi) for lo, hi in eeg.BANDS.values()], axis=-1)     # (ep, ch, 5)
    return bp, band(*eeg.BANDPASS) + 1e-12


def region_of(canon, scheme=""):
    r = eeg.channel_region(canon)
    if "p78T" in scheme and canon in ("P7", "P8"):
        r = "T_L" if canon == "P7" else "T_R"
    if "splitmid" in scheme and r == "MID":
        pre = re.match(r"[A-Z]+", canon).group(0)
        r = "MID_F" if pre in ("FP", "AF", "F") else "MID_C" if pre in ("FC", "C") else "MID_P"
    return r


def region_list(scheme=""):
    regs = [r for r in eeg.REGIONS if r != "MID"]
    return regs + (["MID_F", "MID_C", "MID_P"] if "splitmid" in scheme else ["MID"])


def region_features(epochs, canon, bp=None, power="rel", scheme=""):
    """Same layout as eeg.epoch_features (77 values) for the default scheme with relative power."""
    bp, total = bp if bp is not None else band_powers(epochs)
    eps = 1e-6
    vals = bp / total[..., None] if power == "rel" else np.log10(bp + 1e-12)
    regs = np.array([region_of(c, scheme) for c in canon])

    def block(v, raw):
        tbr = (np.log10((v[:, 1] + eps) / (v[:, 3] + eps)) if power == "rel"
               else np.log10((raw[:, 1] + 1e-12) / (raw[:, 3] + 1e-12)))
        return np.column_stack([v, tbr])
    feats, alpha = [], {}
    order = region_list(scheme)
    if not scheme:
        order = eeg.REGIONS
    for r in order:
        idx = np.flatnonzero(regs == r)
        if idx.size:
            v, raw = vals[:, idx, :].mean(axis=1), bp[:, idx, :].mean(axis=1)
        else:
            v = raw = np.full((bp.shape[0], 5), np.nan)
        feats.append(block(v, raw))
        alpha[r] = v[:, 2] if power == "rel" else raw[:, 2]
    feats.append(np.column_stack([np.log10((alpha[f"{lb}_R"] + eps) / (alpha[f"{lb}_L"] + eps)) for lb in eeg.LOBES]))
    feats.append(block(vals.mean(axis=1), bp.mean(axis=1)))
    return np.concatenate(feats, axis=1)


def channel_features(epochs, canon, bp=None, power="rel"):
    """Per-channel features on the 19 10-20 sites (6 each = 114); missing sites are NaN."""
    bp, total = bp if bp is not None else band_powers(epochs)
    vals = bp / total[..., None] if power == "rel" else np.log10(bp + 1e-12)
    out = []
    canon = np.array(canon)
    for c in BASE_CANON:
        idx = np.flatnonzero(canon == c)
        if idx.size:
            v, raw = vals[:, idx, :].mean(axis=1), bp[:, idx, :].mean(axis=1)
            out.append(np.column_stack([v, np.log10((raw[:, 1] + 1e-12) / (raw[:, 3] + 1e-12))]))
        else:
            out.append(np.full((bp.shape[0], 6), np.nan))
    return np.concatenate(out, axis=1)


def spline_region_features(epochs, canon, bp=None):
    """Spherical-spline interpolation of missing 10-20 channels (MNE), then the standard 77 features."""
    if not HAVE_MNE or any(c not in BASE_CANON for c in canon):
        return None
    try:
        return _spline(epochs, canon)
    except Exception:                       # interpolation failure -> treat as unprocessable
        return None


def _spline(epochs, canon):
    full = np.zeros((epochs.shape[0], 19, epochs.shape[2]))
    present = []
    for j, c in enumerate(BASE_CANON):
        idx = [i for i, cc in enumerate(canon) if cc == c]
        if idx:
            full[:, j] = epochs[:, idx].mean(axis=1)
            present.append(j)
    info = mne.create_info(BASE_CH, eeg.TARGET_FS, "eeg")
    ep = mne.EpochsArray(full * 1e-6, info, verbose=False)
    ep.set_montage("standard_1020")
    ep.info["bads"] = [BASE_CH[j] for j in range(19) if j not in present]
    if ep.info["bads"]:
        ep.interpolate_bads(reset_bads=True, verbose=False)
    return eeg.epoch_features(ep.get_data() * 1e6, BASE_CANON)


FEATURE_FNS = {
    "region_rel": lambda e, c, bp: region_features(e, c, bp, "rel"),
    "region_abs": lambda e, c, bp: region_features(e, c, bp, "abs"),
    "region_rel_splitmid": lambda e, c, bp: region_features(e, c, bp, "rel", "splitmid"),
    "region_rel_p78T": lambda e, c, bp: region_features(e, c, bp, "rel", "p78T"),
    "channel_rel": lambda e, c, bp: channel_features(e, c, bp, "rel"),
    "channel_abs": lambda e, c, bp: channel_features(e, c, bp, "abs"),
    "region_rel_spline": lambda e, c, bp: spline_region_features(e, c, bp),
}
CHANNEL_COLS = {c: list(range(6 * j, 6 * j + 6)) for j, c in enumerate(BASE_CANON)}


# ----------------------------------------------------------------------------- cohort workers
def _subject_worker(task):
    rng = np.random.default_rng(task["ss"])
    if task["sim"] == "v1":
        sig, info = simulate_v1(task["label"], task["canon"], task["fs"], task["dur"], rng, task["effect"])
    else:
        sig, info = simulate_v2(task["label"], task["sensors"], task["fs"], task["dur"], rng,
                                task["effect"], net=task["net"], p=task.get("p", SIM2))
    if task["net"] == "egi":
        names, style = list(EGI_NAMES), "egi"
    else:
        names, style = style_names(task["canon"], rng, task.get("style"))
    out = {"meta": dict(sid=task["sid"], label=task["label"], kind=task["kind"], fs=task["fs"], dur=task["dur"],
                        nch=len(names), n_regions=n_regions(task["canon"]) if task["net"] != "egi" else 11,
                        style=style, **{k: v for k, v in info.items() if k != "bad_idx"}),
           "feats": {}}
    bad_name = None if info["bad_idx"] is None else names[info["bad_idx"]]
    for pname, kinds in task["plan"].items():
        try:
            epochs, canon, rep = prep(sig, task["fs"], names, PREP_CFGS[pname], task.get("site_dist"))
            bp = band_powers(epochs)
            for k in kinds:
                out["feats"][f"{pname}|{k}"] = FEATURE_FNS[k](epochs, canon, bp)
            if pname == "full":
                out["meta"].update(
                    det_unit=rep["units"].split()[0], det_mains=rep["mains_hz"],
                    det_bad=[names[i] for i in rep["bad_idx"]], inj_bad=bad_name,
                    n_epochs=rep["n_epochs"], regions_used=rep["regions"], n_mapped=rep["n_mapped"])
        except ValueError as err:
            for k in kinds:
                out["feats"][f"{pname}|{k}"] = None
            if pname == "full":
                out["meta"]["error"] = str(err)
    return out


def _multi_montage_worker(task):
    """E7/E13: one subject recorded with all 19 channels, then re-analysed under many channel subsets."""
    rng = np.random.default_rng(task["ss"])
    sig, info = simulate_v2(task["label"], BASE_CANON, task["fs"], task["dur"], rng, task["effect"],
                            p=task.get("p", SIM2))
    names = list(BASE_CH)
    out = []
    for mname, chans in task["montages"]:
        idx = [BASE_CANON.index(c) for c in chans]
        if n_regions(chans) < 4:                     # the pipeline refuses these montages
            out.append(dict(sid=task["sid"], label=task["label"], montage=mname, n_ch=len(chans),
                            n_regions=n_regions(chans), x=None))
            continue
        try:
            epochs, canon, rep = prep(sig[idx], task["fs"], [names[i] for i in idx], PREP_CFGS["full"])
            x = region_features(epochs, canon)
            regs = rep["regions"]
        except ValueError:
            x, regs = None, n_regions(chans)
        out.append(dict(sid=task["sid"], label=task["label"], montage=mname, n_ch=len(chans),
                        n_regions=regs, x=x))
    return out


def _psd_worker(task):
    """E11: mean preprocessed spectrum (global and left-parietal) for one simulated subject."""
    rng = np.random.default_rng(task["ss"])
    if task["sim"] == "v1":
        sig, _ = simulate_v1(task["label"], task["canon"], task["fs"], task["dur"], rng, task["effect"])
    else:
        sig, _ = simulate_v2(task["label"], task["canon"], task["fs"], task["dur"], rng, task["effect"],
                             p=task.get("p", SIM2))
    names, _ = style_names(task["canon"], rng, "plain")
    epochs, canon, _ = prep(sig, task["fs"], names, PREP_CFGS["full"])
    freqs, psd = signal.welch(epochs, fs=eeg.TARGET_FS, nperseg=int(2 * eeg.TARGET_FS), axis=-1)
    psd = psd.mean(axis=0)                                           # (ch, f)
    psd = psd / psd[:, (freqs >= 1) & (freqs <= 45)].sum(axis=1, keepdims=True)   # relative spectrum
    lp = [i for i, c in enumerate(canon) if eeg.channel_region(c) == "P_L"]
    return dict(label=task["label"], kind=task["kind"], freqs=freqs, glob=psd.mean(axis=0),
                pl=psd[lp].mean(axis=0) if lp else None)


In [ ]:
import revision_sim as rs
importlib.reload(rs)
from revision_sim import (BASE_CH, BASE_CANON, OPENBCI8, OPENBCI8_CANON, EGI_NAMES, SIM2, PREP_CFGS,
                          CHANNEL_COLS, prep, region_features, simulate_v2, style_names, n_regions)

(RES / "simulator_v2_parameters.json").write_text(json.dumps(SIM2, indent=2))
_d = np.linalg.norm(rs.SRC_XYZ[:, None] - rs.SRC_XYZ[None], axis=-1)
print(f"median nearest-neighbour source distance on the unit sphere: {np.median(np.sort(_d, 1)[:, 1]):.2f} "
      f"(volume-conduction kernel width {SIM2['vc_lambda']})")

# equivalence check against the original module (must print True)
_rng = np.random.default_rng(0)
_sig, _ = simulate_v2(1, BASE_CANON, 500, 32, _rng, 0.3)
_names, _ = style_names(BASE_CANON, _rng, "ref")
_e1, _c1, _ = eeg.preprocess_recording(_sig, 500, _names)
_e2, _c2, _ = prep(_sig, 500, _names, PREP_CFGS["full"])
print("configurable pipeline == eeg.preprocess_recording:",
      np.allclose(eeg.epoch_features(_e1, _c1), region_features(_e2, _c2), equal_nan=True))

## 2. Cohort generation (parallel, cached, reproducible)
Every subject has its own seed, derived from (cohort seed, subject index). Results therefore do not depend
on the number of cores. Each subject is generated once and passed through every requested pipeline variant.

In [ ]:
def _cache_key(**kw):
    return hashlib.md5(json.dumps(kw, sort_keys=True, default=str).encode()).hexdigest()[:16]


def build_cohort(seed, sim, effect, plan, n=N_SUBJECTS, kind=None, net="1020", fs_choices=FS_CHOICES,
                 durations=DURATION_CHOICES_S, sid_offset=0, site_dist=None, tag=""):
    """Returns dict(meta=DataFrame, feats={variant: {sid: (n_epochs, n_feat) array or None}})."""
    key = _cache_key(seed=seed, sim=sim, effect=effect, plan=plan, n=n, kind=kind, net=net, fs=fs_choices,
                     dur=durations, off=sid_offset, sd=site_dist, sim2=SIM2 if sim == "v2" else None, v=3)
    path = CACHE / f"cohort_{tag}{sim}_e{effect:.2f}_s{seed}_{key}.pkl"
    if path.exists():
        return pickle.loads(path.read_bytes())
    rng = np.random.default_rng([seed, 7919])
    labels = np.array([0] * (n // 2) + [1] * (n - n // 2))
    rng.shuffle(labels)
    tasks = []
    for i in range(n):
        if net == "egi":
            k, canon = "egi128", list(BASE_CANON)
        else:
            k, canon = rs.random_montage(rng, kind, MONTAGE_P)
        tasks.append(dict(sid=sid_offset + i, label=int(labels[i]), kind=k, canon=canon,
                          sensors=EGI_NAMES if net == "egi" else canon, net=net, sim=sim, effect=effect,
                          fs=int(rng.choice(fs_choices)), dur=int(rng.choice(durations)), plan=plan,
                          site_dist=site_dist, p=SIM2, ss=np.random.SeedSequence([seed, 104729, i]).entropy))
    t0 = time.time()
    outs = Parallel(n_jobs=N_JOBS)(delayed(rs._subject_worker)(t) for t in tasks)
    meta = pd.DataFrame([o["meta"] for o in outs]).set_index("sid")
    feats = {}
    for o in outs:
        for v, x in o["feats"].items():
            feats.setdefault(v, {})[o["meta"]["sid"]] = x
    coh = dict(meta=meta, feats=feats, seed=seed, sim=sim, effect=effect)
    path.write_bytes(pickle.dumps(coh))
    print(f"  built {tag}{sim} eps={effect:.2f} seed={seed} n={n} kind={kind or 'mix'} in {time.time() - t0:.0f}s")
    return coh


PLAN_REF = {"full": ["region_rel"]}

## 3. Evaluation helpers
Subject = decision unit. A subject's probability is the mean of its epoch probabilities. Folds are assigned
per subject and stratified by class, and the same folds are used for every variant, so comparisons are
paired. A recording that a pipeline variant cannot process gets p = 0.5 in the *intention-to-test* (ITT)
metrics, which counts it as an error for accuracy.

In [ ]:
def make_model(name="RF", seed=0):
    imp = SimpleImputer(strategy="median", keep_empty_features=True)
    sc = StandardScaler()
    if name == "RF":
        clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced",
                                     n_jobs=N_JOBS_MODEL, random_state=seed)
    elif name == "RF_ind":
        imp = SimpleImputer(strategy="median", keep_empty_features=True, add_indicator=True)
        clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced",
                                     n_jobs=N_JOBS_MODEL, random_state=seed)
    elif name == "HGB":
        return HistGradientBoostingClassifier(class_weight="balanced", random_state=seed)
    elif name == "GB":
        clf = GradientBoostingClassifier(random_state=seed)
    elif name == "LR":
        clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0)
    elif name == "SVM":
        clf = SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=seed)
    elif name == "LDA":
        clf = LinearDiscriminantAnalysis()
    elif name == "kNN":
        clf = KNeighborsClassifier(n_neighbors=15)
    else:
        raise ValueError(name)
    return Pipeline([("impute", imp), ("scale", sc), ("clf", clf)])


def stack(coh, variant, sids=None, cols=None):
    """Epoch matrix for the given subjects. Subjects whose features are None, or NaN in `cols`, are skipped."""
    fd = coh["feats"][variant]
    sids = coh["meta"].index if sids is None else sids
    X, y, g = [], [], []
    for s in sids:
        x = fd.get(s)
        if x is None:
            continue
        if cols is not None:
            x = x[:, cols]
            if np.isnan(x).any():
                continue
        X.append(x)
        y += [int(coh["meta"].at[s, "label"])] * len(x)
        g += [s] * len(x)
    if not X:
        return np.empty((0, 0)), np.array([]), np.array([])
    return np.vstack(X), np.array(y), np.array(g)


def subject_mean(p, g):
    s = pd.Series(p).groupby(g).mean()
    return s


def subject_folds(meta, seed, k=K_FOLDS):
    sids = meta.index.to_numpy()
    skf = StratifiedKFold(k, shuffle=True, random_state=seed)
    fold = pd.Series(-1, index=sids)
    for f, (_, te) in enumerate(skf.split(sids, meta["label"].to_numpy())):
        fold.iloc[te] = f
    return fold


def oof_subject_probs(coh, variant, folds, model="RF", seed=0, cols=None, return_times=False):
    """Out-of-fold subject probabilities (NaN = not processable)."""
    X, y, g = stack(coh, variant, cols=cols)
    p_subj = pd.Series(np.nan, index=coh["meta"].index)
    times = []
    if len(X) == 0:
        return (p_subj, times) if return_times else p_subj
    fg = folds.loc[g].to_numpy()
    for f in sorted(folds.unique()):
        tr, te = fg != f, fg == f
        if te.sum() == 0 or len(np.unique(y[tr])) < 2:
            continue
        m = make_model(model, seed)
        t0 = time.time()
        m.fit(X[tr], y[tr])
        times.append(time.time() - t0)
        sp = subject_mean(m.predict_proba(X[te])[:, 1], g[te])
        p_subj.loc[sp.index] = sp.to_numpy()
    return (p_subj, times) if return_times else p_subj


def fit_predict(coh_tr, sids_tr, coh_te, sids_te, variant="full|region_rel", model="RF", seed=0, cols=None,
                return_epoch=False):
    Xtr, ytr, _ = stack(coh_tr, variant, sids_tr, cols)
    Xte, yte, gte = stack(coh_te, variant, sids_te, cols)
    m = make_model(model, seed).fit(Xtr, ytr)
    pe = m.predict_proba(Xte)[:, 1]
    ps = pd.Series(np.nan, index=pd.Index(sids_te))
    sp = subject_mean(pe, gte)
    ps.loc[sp.index] = sp.to_numpy()
    if return_epoch:
        return ps, (pe, yte, gte), m
    return ps


def sens_at_spec(y, p, spec=0.90):
    fpr, tpr, _ = roc_curve(y, p)
    ok = fpr <= 1 - spec + 1e-12
    return float(tpr[ok].max()) if ok.any() else 0.0


def ece(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    return float(sum(abs(y[idx == b].mean() - p[idx == b].mean()) * (idx == b).mean()
                     for b in range(bins) if (idx == b).any()))


def metrics(y, p, thr=0.5, itt=True):
    """Subject-level metrics. NaN probabilities = unprocessable -> 0.5 (ITT) or dropped."""
    y, p = np.asarray(y, int), np.asarray(p, float)
    missing = ~np.isfinite(p)
    cover = float(1 - missing.mean())
    if itt:
        p = np.where(missing, 0.5, p)
    else:
        y, p, missing = y[~missing], p[~missing], missing[~missing]
    pred = (p > thr).astype(int)
    pred = np.where(missing, 1 - y, pred)                # unprocessable counted as an error (ITT)
    out = dict(n=len(y), coverage=cover, acc=accuracy_score(y, pred), bacc=balanced_accuracy_score(y, pred),
               sens=float((pred[y == 1] == 1).mean()) if (y == 1).any() else np.nan,
               spec=float((pred[y == 0] == 0).mean()) if (y == 0).any() else np.nan)
    if len(np.unique(y)) == 2:
        out.update(auc=roc_auc_score(y, p), sens90=sens_at_spec(y, p), brier=brier_score_loss(y, p), ece=ece(y, p))
    return out


def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    ph = k / n
    d = 1 + z * z / n
    c = (ph + z * z / (2 * n)) / d
    h = z * np.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


def msd(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    return f"{x.mean():.3f} ± {x.std(ddof=1) if len(x) > 1 else 0:.3f}" if len(x) else "n/a"


def cohens_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    sp = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / sp


GLOBAL_LOGTBR = eeg.FEATURE_NAMES.index("global_logTBR")
GLOBAL_THETA = eeg.FEATURE_NAMES.index("global_theta")


def subject_feature(coh, col, variant="full|region_rel"):
    return pd.Series({s: np.nanmean(x[:, col]) for s, x in coh["feats"][variant].items() if x is not None})


VAR = "full|region_rel"
KINDS = ["full19", "openbci8", "random"]


def power_change(eps, w=1.0):
    """Percentage change in band power implied by the amplitude effect (theta, alpha/beta)."""
    return 100 * ((1 + eps * w) ** 2 - 1), 100 * ((1 - 0.5 * eps * w) ** 2 - 1)


log("RUN CONFIGURATION", f"{HARDWARE}\nQUICK={QUICK} N_SUBJECTS={N_SUBJECTS} SEEDS_MAIN={SEEDS_MAIN} "
    f"SEEDS_SWEEP={SEEDS_SWEEP} SEEDS_SMALL={SEEDS_SMALL}\nEFFECT_GRID={EFFECT_GRID} "
    f"ORIGINAL_EFFECT={ORIGINAL_EFFECT} MAIN_EFFECT={MAIN_EFFECT}\n"
    f"power change at MAIN_EFFECT: theta {power_change(MAIN_EFFECT)[0]:+.0f}% / alpha,beta "
    f"{power_change(MAIN_EFFECT)[1]:+.0f}% (x1.6 regions: {power_change(MAIN_EFFECT, 1.6)[0]:+.0f}% / "
    f"{power_change(MAIN_EFFECT, 1.6)[1]:+.0f}%)")

## E11. Example simulated spectra (run first as a sanity check)
These are mean relative spectra after preprocessing, by class and montage, for the original v1 simulator and
the realistic v2 simulator. The second figure shows individual v2 subjects to make the biological
variability visible.

In [ ]:
if RUN["E11"]:
    tasks = []
    for sim, eps in [("v1", ORIGINAL_EFFECT), ("v2", MAIN_EFFECT)]:
        r = np.random.default_rng(11)
        for kind in KINDS:
            for lab in (0, 1):
                for i in range(10 if QUICK else 30):
                    k, canon = rs.random_montage(r, kind)
                    tasks.append(dict(sim=sim, effect=eps, kind=kind, label=lab, canon=canon, fs=250, dur=40, p=SIM2,
                                      ss=np.random.SeedSequence([11, len(tasks)]).entropy))
    psd_out = Parallel(n_jobs=N_JOBS)(delayed(rs._psd_worker)(t) for t in tasks)
    for t, o in zip(tasks, psd_out):
        o["sim"] = t["sim"]
    freqs = psd_out[0]["freqs"]
    fm = (freqs >= 1) & (freqs <= 45)
    fig, axes = plt.subplots(2, 3, figsize=(10, 5.4), sharex=True)
    for i, sim in enumerate(["v1", "v2"]):
        for j, kind in enumerate(KINDS):
            ax = axes[i, j]
            for lab, col, nm in [(0, C_CTRL, "Control"), (1, C_DYS, "Dyslexic")]:
                S = np.array([o["glob"] for o in psd_out if o["sim"] == sim and o["kind"] == kind and o["label"] == lab])
                med, lo, hi = np.median(S, 0), np.percentile(S, 25, 0), np.percentile(S, 75, 0)
                ax.semilogy(freqs[fm], med[fm], color=col, lw=1.6, label=nm)
                ax.fill_between(freqs[fm], lo[fm], hi[fm], color=col, alpha=0.18, lw=0)
            for lo_, hi_ in eeg.BANDS.values():
                ax.axvline(lo_, color="0.85", lw=0.6, zorder=0)
            ax.set_title(f"{sim} (ε={ORIGINAL_EFFECT if sim == 'v1' else MAIN_EFFECT}) - {kind}", fontsize=9)
            if j == 0:
                ax.set_ylabel("relative power (all channels)")
            if i == 1:
                ax.set_xlabel("Frequency (Hz)")
    axes[0, 0].legend(frameon=False)
    fig.tight_layout()
    savefig(fig, "E11_example_spectra")
    plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
    for a, lab, nm, col in [(ax[0], 0, "Control", C_CTRL), (ax[1], 1, "Dyslexic", C_DYS)]:
        S = [o["pl"] for o in psd_out if o["sim"] == "v2" and o["kind"] == "full19" and o["label"] == lab]
        for s_ in S[:15]:
            a.semilogy(freqs[fm], s_[fm], color=col, lw=0.6, alpha=0.6)
        a.set_title(f"v2 individual subjects - left parietal - {nm}", fontsize=9)
        a.set_xlabel("Frequency (Hz)")
    ax[0].set_ylabel("relative power")
    fig.tight_layout()
    savefig(fig, "E11_v2_individual_spectra")
    plt.show()

## E1. Effect-size sweep (review: top-5 #1)
For each simulator, effect size ε and seed, a new cohort is generated and evaluated with subject-level
5-fold CV. ε = 0 is a **null check**: accuracy must be about 0.5, otherwise something leaks. Cohen's d of
the subject-mean global log θ/β gives an effect-size scale that does not depend on the simulator.

In [ ]:
if RUN["E1"]:
    rows = []
    for sim in ("v1", "v2"):
        for eps in EFFECT_GRID:
            for seed in SEEDS_SWEEP:
                coh = build_cohort(seed, sim, eps, PLAN_REF)
                meta = coh["meta"]
                p = oof_subject_probs(coh, VAR, subject_folds(meta, seed), seed=seed)
                m = metrics(meta["label"], p)
                tbr, th = subject_feature(coh, GLOBAL_LOGTBR), subject_feature(coh, GLOBAL_THETA)
                y = meta["label"].loc[tbr.index]
                row = dict(sim=sim, eps=eps, seed=seed, **m, d_logTBR=cohens_d(tbr[y == 1], tbr[y == 0]),
                           d_theta=cohens_d(th[y == 1], th[y == 0]),
                           theta_pow_pct=power_change(eps)[0], alpha_pow_pct=power_change(eps)[1])
                for k in KINDS:
                    sel = meta["kind"] == k
                    row[f"acc_{k}"] = metrics(meta["label"][sel], p[sel])["acc"]
                rows.append(row)
    e1 = pd.DataFrame(rows)
    e1.to_csv(RES / "E1_effect_size_sweep_raw.csv", index=False)
    cols = ["acc", "auc", "sens90", "brier", "d_logTBR", "acc_full19", "acc_openbci8", "acc_random"]
    e1_tab = e1.groupby(["sim", "eps"]).agg(
        theta_pow=("theta_pow_pct", "first"), alpha_pow=("alpha_pow_pct", "first"),
        **{c: (c, msd) for c in cols}).reset_index()
    e1_tab.to_csv(RES / "E1_effect_size_sweep.csv", index=False)
    log("E1 Effect-size sweep: subject-level 5-fold CV (mean ± SD over seeds); theta/alpha power change in %", e1_tab)

    fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
    for sim, col, mk in [("v1", "0.45", "s"), ("v2", C_DYS, "o")]:
        d = e1[e1.sim == sim].groupby("eps")
        for a, met in [(ax[0], "acc"), (ax[1], "auc")]:
            mu, sd = d[met].mean(), d[met].std()
            a.plot(mu.index, mu, marker=mk, color=col, label=f"{sim} ({'original' if sim == 'v1' else 'realistic'})")
            a.fill_between(mu.index, mu - sd, mu + sd, color=col, alpha=0.2, lw=0)
        ax[2].scatter(e1[e1.sim == sim]["d_logTBR"], e1[e1.sim == sim]["auc"], color=col, marker=mk, s=14, label=sim)
    for a, lab in [(ax[0], "Subject accuracy"), (ax[1], "Subject ROC-AUC")]:
        a.axhline(0.5, ls="--", color="k", lw=0.7)
        a.axvline(ORIGINAL_EFFECT, ls=":", color="0.5", lw=0.8)
        a.set(xlabel="Simulated effect size ε", ylabel=lab, ylim=(0.35, 1.02))
    ax[0].legend(frameon=False, fontsize=8)
    ax[2].axhline(0.5, ls="--", color="k", lw=0.7)
    ax[2].set(xlabel="Cohen's d (global log θ/β)", ylabel="Subject ROC-AUC", ylim=(0.35, 1.02))
    ax[2].legend(frameon=False, fontsize=8)
    fig.tight_layout()
    savefig(fig, "E1_effect_size_sweep")
    plt.show()

## E2. Whole pipeline over 10 seeds (generation included)
Three configurations are evaluated: the paper's original setting (v1, ε = 0.30), the same effect under the
realistic simulator (v2, ε = 0.30), and the revision's main setting (v2, `MAIN_EFFECT`). The metrics are
(a) the paper's 75/25 subject hold-out, repeated with a new cohort and split for every seed;
(b) subject-level 5-fold CV, which the review requested; and (c) for the main setting, a decision threshold
tuned by Youden's J **inside the training folds** instead of the fixed 0.5.

In [ ]:
E2_CONFIGS = [("v1", ORIGINAL_EFFECT), ("v2", ORIGINAL_EFFECT), ("v2", MAIN_EFFECT)]
OOF = {}               # (sim, eps, seed) -> DataFrame of out-of-fold subject probabilities + metadata


def get_oof(sim, eps, seed):
    if (sim, eps, seed) not in OOF:
        coh = build_cohort(seed, sim, eps, PLAN_REF)
        d = coh["meta"].copy()
        d["p"] = oof_subject_probs(coh, VAR, subject_folds(d, seed), seed=seed)
        OOF[(sim, eps, seed)] = d
    return OOF[(sim, eps, seed)]


def tuned_threshold_predictions(coh, folds, seed):
    meta = coh["meta"]
    pred = pd.Series(np.nan, index=meta.index)
    thrs = []
    for f in sorted(folds.unique()):
        tr, te = folds.index[folds != f], folds.index[folds == f]
        sub = dict(meta=meta.loc[tr], feats=coh["feats"])
        p_in = oof_subject_probs(sub, VAR, subject_folds(sub["meta"], seed + 100, k=3), seed=seed)
        fpr, tpr, thr = roc_curve(meta.loc[tr, "label"], p_in)
        t = float(np.clip(thr[np.argmax(tpr - fpr)], 0.05, 0.95))
        thrs.append(t)
        p_te = fit_predict(coh, tr, coh, te, seed=seed)
        pred.loc[te] = (p_te >= t).astype(float)
    return pred, thrs


if RUN["E2"]:
    rows = []
    for sim, eps in E2_CONFIGS:
        for seed in SEEDS_MAIN:
            coh = build_cohort(seed, sim, eps, PLAN_REF)
            meta = coh["meta"]
            sids = meta.index.to_numpy()
            tr, te = next(GroupShuffleSplit(1, test_size=0.25, random_state=seed).split(sids, groups=sids))
            ps, (pe, ye, ge), _ = fit_predict(coh, sids[tr], coh, sids[te], seed=seed, return_epoch=True)
            mh = metrics(meta.loc[sids[te], "label"], ps)
            d = get_oof(sim, eps, seed)
            mo = metrics(d["label"], d["p"])
            row = dict(sim=sim, eps=eps, seed=seed, ho_epoch_acc=accuracy_score(ye, pe > 0.5),
                       ho_epoch_auc=roc_auc_score(ye, pe), **{f"ho_{k}": v for k, v in mh.items()},
                       **{f"cv_{k}": v for k, v in mo.items()})
            for k in KINDS:
                sel = d["kind"] == k
                row[f"cv_acc_{k}"] = metrics(d["label"][sel], d["p"][sel])["acc"]
            if (sim, eps) == ("v2", MAIN_EFFECT):
                pred, thrs = tuned_threshold_predictions(coh, subject_folds(meta, seed), seed)
                yy = meta["label"]
                row.update(tuned_thr_mean=np.mean(thrs), tuned_acc=accuracy_score(yy, pred),
                           tuned_sens=float((pred[yy == 1] == 1).mean()), tuned_spec=float((pred[yy == 0] == 0).mean()))
            rows.append(row)
    e2 = pd.DataFrame(rows)
    e2.to_csv(RES / "E2_multiseed_raw.csv", index=False)
    show = ["ho_epoch_acc", "ho_epoch_auc", "ho_acc", "ho_auc", "ho_sens", "ho_spec", "cv_acc", "cv_auc", "cv_sens",
            "cv_spec", "cv_sens90", "cv_brier", "cv_ece", "cv_acc_full19", "cv_acc_openbci8", "cv_acc_random",
            "tuned_thr_mean", "tuned_acc", "tuned_sens", "tuned_spec"]
    e2_tab = e2.groupby(["sim", "eps"])[show].agg(msd).T
    e2_tab.to_csv(RES / "E2_multiseed_summary.csv")
    log(f"E2 Whole pipeline repeated over {len(SEEDS_MAIN)} seeds (mean ± SD). ho_ = 75/25 hold-out (90 subj), "
        "cv_ = subject 5-fold CV (all subjects); tuned_ = threshold tuned inside training folds", e2_tab)
    rng_tab = e2.groupby(["sim", "eps"]).agg(ho_acc_min=("ho_acc", "min"), ho_acc_max=("ho_acc", "max"),
                                             ho_auc_min=("ho_auc", "min"), ho_auc_max=("ho_auc", "max"))
    log("E2 Seed-to-seed range of the single 75/25 hold-out (the paper reported one split: acc 0.978, AUC 1.000)", rng_tab)

## E10. Auto-detection counts (review 5A)
These counts are pooled over all E2 cohorts. For v2, false bad-channel flags are also broken down by
whether a blink or EMG artefact was present, because those artefacts are realistic sources of false alarms.

In [ ]:
if RUN["E10"]:
    det_rows = []
    for sim, eps in [("v1", ORIGINAL_EFFECT), ("v2", MAIN_EFFECT)]:
        M = pd.concat([build_cohort(s, sim, eps, PLAN_REF)["meta"] for s in SEEDS_MAIN])
        M = M[M["det_unit"].notna()]
        hasbad = M["inj_bad"].notna()
        caught = M[hasbad].apply(lambda r: r["inj_bad"] in r["det_bad"], axis=1)
        extra = M.apply(lambda r: len([b for b in r["det_bad"] if b != r["inj_bad"]]), axis=1)
        mains = M["mains"].fillna(0)
        detm = M["det_mains"].fillna(0)
        for what, k, n in [
            ("units stored in V -> detected V", ((M.unit == "V") & (M.det_unit == "V")).sum(), (M.unit == "V").sum()),
            ("units stored in uV -> detected uV", ((M.unit == "uV") & (M.det_unit == "uV")).sum(), (M.unit == "uV").sum()),
            ("50 Hz mains -> detected 50", ((mains == 50) & (detm == 50)).sum(), (mains == 50).sum()),
            ("60 Hz mains -> detected 60", ((mains == 60) & (detm == 60)).sum(), (mains == 60).sum()),
            ("no mains -> none detected (no false alarm)", ((mains == 0) & (detm == 0)).sum(), (mains == 0).sum()),
            ("flat channel injected -> caught", caught[M[hasbad].bad_type == "flat"].sum(), (M[hasbad].bad_type == "flat").sum()),
            ("noisy channel injected -> caught", caught[M[hasbad].bad_type == "noisy"].sum(), (M[hasbad].bad_type == "noisy").sum()),
            ("recordings with >=1 false bad-channel flag", (extra > 0).sum(), len(M)),
            ("  ... of which with blink artefact", ((extra > 0) & M.blink.astype(bool)).sum(), M.blink.astype(bool).sum()),
            ("  ... of which with EMG artefact", ((extra > 0) & M.emg.astype(bool)).sum(), M.emg.astype(bool).sum()),
        ]:
            lo, hi = wilson(k, n)
            det_rows.append(dict(sim=sim, check=what, k=int(k), n=int(n), rate=k / n if n else np.nan, ci_lo=lo, ci_hi=hi))
    e10 = pd.DataFrame(det_rows)
    e10.to_csv(RES / "E10_detection_counts.csv", index=False)
    log("E10 Auto-detection counts (pooled over E2 cohorts)", e10.round(3))

## E6. Calibration and the confidence score (review 5B)

In [ ]:
if RUN["E6"]:
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
    cal_rows = []
    for (sim, eps), col in zip(E2_CONFIGS, ["0.45", C_ACC, C_DYS]):
        D = pd.concat([get_oof(sim, eps, s) for s in SEEDS_MAIN])
        y, p = D["label"].to_numpy(), D["p"].to_numpy()
        fr, mp = calibration_curve(y, p, n_bins=10, strategy="quantile")
        ax[0].plot(mp, fr, marker="o", ms=3, color=col, label=f"{sim} ε={eps}")
        conf = np.maximum(p, 1 - p)
        correct = (p > 0.5) == y
        order = np.argsort(-conf)
        cov = np.arange(1, len(p) + 1) / len(p)
        ax[1].plot(cov, np.cumsum(correct[order]) / np.arange(1, len(p) + 1), color=col, label=f"{sim} ε={eps}")
        ab = {f"acc_keep{int(100 * q)}": correct[order][: int(q * len(p))].mean() for q in (1.0, 0.9, 0.8, 0.5)}
        cal_rows.append(dict(sim=sim, eps=eps, n=len(p), brier=brier_score_loss(y, p), ece=ece(y, p),
                             auc_conf_vs_correct=roc_auc_score(correct, conf) if 0 < correct.mean() < 1 else np.nan,
                             conf_correct_median=np.median(conf[correct]),
                             conf_wrong_median=np.median(conf[~correct]) if (~correct).any() else np.nan, **ab))
        if (sim, eps) == ("v2", MAIN_EFFECT):
            ax[2].boxplot([conf[correct], conf[~correct]], widths=0.5)
            ax[2].set_xticks([1, 2], ["correct", "incorrect"])
            ax[2].set(ylabel="confidence max(p, 1-p)", title=f"v2 ε={eps}")
    ax[0].plot([0, 1], [0, 1], "k--", lw=0.7)
    ax[0].set(xlabel="mean predicted P(dyslexic)", ylabel="observed fraction dyslexic", title="Reliability (pooled OOF)")
    ax[1].set(xlabel="fraction of subjects retained (most confident first)", ylabel="accuracy",
              title="Selective prediction")
    ax[0].legend(frameon=False, fontsize=7)
    fig.tight_layout()
    savefig(fig, "E6_calibration")
    plt.show()
    e6 = pd.DataFrame(cal_rows)
    e6.to_csv(RES / "E6_calibration.csv", index=False)
    log("E6 Calibration and confidence (pooled out-of-fold subject probabilities over seeds)", e6.round(3))

## E3. Generalisation to unseen acquisition set-ups (review: top-5 #2)
**Leave-one-montage-out (LMO).** The test set is half of the subjects of montage type *k* (both halves are
used in turn). The LMO model is trained on all subjects of the *other* montage types. The control model is
trained on the **same number** of subjects drawn from everyone outside the test half, so montage *k* has
been seen in training. Both models are tested on the same subjects, which makes the comparison paired.
The same design is used for leave-one-sampling-rate-out. **EGI-128.** A model trained on the 10-20
cohort (8–19 channels) is tested on a separately simulated 128-channel EGI cohort, which reaches the
pipeline only through the 3-D position route.

In [ ]:
def heldout_vs_seen(coh, mask_k, seed, reps=2):
    """Paired comparison on the same test subjects: model without group k vs size-matched model with it."""
    meta = coh["meta"]
    r = np.random.default_rng(seed)
    out = []
    ks = meta.index[mask_k].to_numpy()
    others = meta.index[~mask_k].to_numpy()
    if len(ks) < 6:
        return out
    for rep in range(reps):
        perm = r.permutation(ks)
        halves = [perm[: len(perm) // 2], perm[len(perm) // 2:]]
        for T in halves:
            pool = np.setdiff1d(meta.index.to_numpy(), T)
            seen_tr = r.choice(pool, size=min(len(others), len(pool)), replace=False)
            p_un = fit_predict(coh, others, coh, T, seed=seed)
            p_se = fit_predict(coh, seen_tr, coh, T, seed=seed)
            y = meta.loc[T, "label"]
            out.append(dict(n_test=len(T), n_train=len(others), **{f"unseen_{k}": v for k, v in metrics(y, p_un).items()},
                            **{f"seen_{k}": v for k, v in metrics(y, p_se).items()}))
        if QUICK:
            break
    return out


if RUN["E3"]:
    rows = []
    for sim, eps in [("v1", ORIGINAL_EFFECT), ("v2", MAIN_EFFECT)]:
        for seed in SEEDS_SWEEP:
            coh = build_cohort(seed, sim, eps, PLAN_REF)
            meta = coh["meta"]
            for k in KINDS:
                for r_ in heldout_vs_seen(coh, (meta["kind"] == k).to_numpy(), seed):
                    rows.append(dict(sim=sim, eps=eps, seed=seed, test="leave-montage-out", held_out=k, **r_))
            for fs in FS_CHOICES:
                for r_ in heldout_vs_seen(coh, (meta["fs"] == fs).to_numpy(), seed, reps=1):
                    rows.append(dict(sim=sim, eps=eps, seed=seed, test="leave-fs-out", held_out=str(fs), **r_))
            if sim == "v2" and HAVE_MNE:
                egi = build_cohort(seed + 500, "v2", eps, PLAN_REF, n=N_EGI, net="egi", sid_offset=100000, tag="egi_")
                p = fit_predict(coh, meta.index, egi, egi["meta"].index, seed=seed)
                m = metrics(egi["meta"]["label"], p)
                d_cv = get_oof(sim, eps, seed)
                rows.append(dict(sim=sim, eps=eps, seed=seed, test="unseen EGI-128 cohort", held_out="egi128",
                                 n_test=len(p), n_train=len(meta), **{f"unseen_{k}": v for k, v in m.items()},
                                 seen_acc=metrics(d_cv["label"], d_cv["p"])["acc"],
                                 seen_auc=metrics(d_cv["label"], d_cv["p"])["auc"],
                                 egi_channels_mapped=egi["meta"]["n_mapped"].mean(),
                                 egi_regions=egi["meta"]["regions_used"].mean()))
    e3 = pd.DataFrame(rows)
    e3.to_csv(RES / "E3_generalisation_raw.csv", index=False)
    e3["delta_acc"] = e3["unseen_acc"] - e3["seen_acc"]
    e3["delta_auc"] = e3["unseen_auc"] - e3["seen_auc"]
    e3_tab = e3.groupby(["sim", "test", "held_out"]).agg(
        n_test=("n_test", "mean"), unseen_acc=("unseen_acc", msd), seen_acc=("seen_acc", msd),
        unseen_auc=("unseen_auc", msd), seen_auc=("seen_auc", msd), delta_acc=("delta_acc", msd),
        delta_auc=("delta_auc", msd))
    tests = []
    for key, d in e3[e3.test != "unseen EGI-128 cohort"].groupby(["sim", "test", "held_out"]):
        g = d.groupby("seed")[["unseen_acc", "seen_acc"]].mean()
        tests.append((*key, stats.ttest_rel(g.unseen_acc, g.seen_acc).pvalue if len(g) > 1 else np.nan))
    e3_tab = e3_tab.join(pd.DataFrame(tests, columns=["sim", "test", "held_out", "p_paired_t"]).set_index(
        ["sim", "test", "held_out"]))
    e3_tab.to_csv(RES / "E3_generalisation.csv")
    log("E3 Unseen vs seen acquisition set-up (same test subjects; size-matched training; mean ± SD over seeds x halves). "
        "For EGI, 'seen' = 5-fold CV accuracy on the 10-20 cohort", e3_tab)
    if "egi_channels_mapped" in e3:
        log("E3 EGI cohort: channels mapped by the 3-D position route (of 128) and regions covered",
            e3[e3.test == "unseen EGI-128 cohort"][["seed", "egi_channels_mapped", "egi_regions"]])

## E4. Ablation and naive baselines (review: top-5 #3, 4B, 4C)
Each pipeline component is removed in turn. The pipeline is also compared with conventional single-montage
designs: (i) per-channel features on a fixed 8-channel montage, where recordings lacking any of the 8
channels cannot be analysed; and (ii) a **naive** pipeline with exact channel labels, no automatic unit,
mains, bad-channel or artefact handling, no average reference, absolute power and no region pooling. A
subject that a variant cannot process gets p = 0.5 and counts as an error (*intention-to-test*, ITT);
`coverage` gives the fraction processed. Folds are identical across variants, so the differences are paired.

In [ ]:
PLAN_ABL = {"full": ["region_rel", "region_abs", "channel_rel", "channel_abs", "region_rel_splitmid",
                     "region_rel_p78T"] + (["region_rel_spline"] if HAVE_MNE else []),
            "no_names": ["region_rel"], "no_unit": ["region_rel", "region_abs"], "no_notch": ["region_rel"],
            "no_bad": ["region_rel"], "no_car": ["region_rel"], "no_reject": ["region_rel"],
            "naive": ["channel_abs"]}
FIXED8 = sum((CHANNEL_COLS[c] for c in OPENBCI8_CANON), [])
VARIANTS = [  # label, feature key, model, column subset
    ("Full pipeline", "full|region_rel", "RF", None),
    ("- channel-name normalisation", "no_names|region_rel", "RF", None),
    ("- unit detection", "no_unit|region_rel", "RF", None),
    ("- mains notch", "no_notch|region_rel", "RF", None),
    ("- bad-channel removal", "no_bad|region_rel", "RF", None),
    ("- common average reference", "no_car|region_rel", "RF", None),
    ("- epoch rejection", "no_reject|region_rel", "RF", None),
    ("absolute instead of relative power", "full|region_abs", "RF", None),
    ("absolute power and no unit detection", "no_unit|region_abs", "RF", None),
    ("channel-level features (no region pooling)", "full|channel_rel", "RF", None),
    ("midline split into F/C/P", "full|region_rel_splitmid", "RF", None),
    ("P7/P8 assigned to temporal", "full|region_rel_p78T", "RF", None),
    ("median imputation + missingness indicators", "full|region_rel", "RF_ind", None),
    ("HistGradientBoosting (native NaN, no imputation)", "full|region_rel", "HGB", None),
    ("spherical-spline interpolation (no NaN)", "full|region_rel_spline", "RF", None),
    ("fixed 8-ch montage, relative power", "full|channel_rel", "RF", FIXED8),
    ("fixed 8-ch montage, absolute power", "full|channel_abs", "RF", FIXED8),
    ("NAIVE: exact labels, no auto-prep, abs power, fixed 8-ch", "naive|channel_abs", "RF", FIXED8),
    ("NAIVE: exact labels, no auto-prep, abs power, 19-ch + imputation", "naive|channel_abs", "RF", None),
]

if RUN["E4"]:
    rows = []
    for sim, eps in [("v2", MAIN_EFFECT), ("v1", ORIGINAL_EFFECT)]:
        for seed in SEEDS_SWEEP:
            coh = build_cohort(seed, sim, eps, PLAN_ABL, tag="abl_")
            meta = coh["meta"]
            folds = subject_folds(meta, seed)
            for label, key, model, cols in VARIANTS:
                if key not in coh["feats"]:
                    continue
                p = oof_subject_probs(coh, key, folds, model=model, seed=seed, cols=cols)
                m = metrics(meta["label"], p)
                row = dict(sim=sim, eps=eps, seed=seed, variant=label, **m,
                           acc_processed=metrics(meta["label"], p, itt=False)["acc"] if np.isfinite(p).any() else np.nan)
                for k in KINDS:
                    sel = meta["kind"] == k
                    row[f"acc_{k}"] = metrics(meta["label"][sel], p[sel])["acc"]
                rows.append(row)
    e4 = pd.DataFrame(rows)
    e4.to_csv(RES / "E4_ablation_raw.csv", index=False)
    ref = e4[e4.variant == "Full pipeline"].set_index(["sim", "seed"])
    e4["d_acc"] = e4.apply(lambda r: r["acc"] - ref.loc[(r["sim"], r["seed"]), "acc"], axis=1)
    e4["d_auc"] = e4.apply(lambda r: r["auc"] - ref.loc[(r["sim"], r["seed"]), "auc"], axis=1)
    pvals = {}
    for (sim, v), d in e4.groupby(["sim", "variant"]):
        pvals[(sim, v)] = (stats.ttest_1samp(d["d_auc"], 0).pvalue if len(d) > 1 and d["d_auc"].std() > 0 else np.nan)
    order = {v[0]: i for i, v in enumerate(VARIANTS)}
    e4_tab = e4.groupby(["sim", "variant"]).agg(
        coverage=("coverage", "mean"), acc_itt=("acc", msd), auc=("auc", msd), d_acc=("d_acc", msd),
        d_auc=("d_auc", msd), acc_processed=("acc_processed", msd),
        acc_full19=("acc_full19", msd), acc_openbci8=("acc_openbci8", msd), acc_random=("acc_random", msd))
    e4_tab["p_dAUC"] = [pvals[i] for i in e4_tab.index]
    e4_tab = e4_tab.reset_index()
    e4_tab["o"] = e4_tab["variant"].map(order)
    e4_tab = e4_tab.sort_values(["sim", "o"]).drop(columns="o").set_index(["sim", "variant"])
    e4_tab.to_csv(RES / "E4_ablation.csv")
    log("E4 Ablation and baselines (subject 5-fold CV, ITT; mean ± SD over seeds; d_ = variant - full pipeline, "
        "paired t-test p on d_AUC)", e4_tab)

    fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
    for ax, sim in zip(axes, ["v2", "v1"]):
        d = e4[e4.sim == sim].groupby("variant")[["d_acc"]].agg(["mean", "std"])["d_acc"]
        d = d.loc[[v[0] for v in VARIANTS if v[0] in d.index]]
        ax.barh(range(len(d)), d["mean"], xerr=d["std"], color=[C_ACC if x >= 0 else C_DYS for x in d["mean"]],
                alpha=0.85, error_kw=dict(lw=0.8))
        ax.axvline(0, color="k", lw=0.7)
        ax.set_yticks(range(len(d)))
        ax.set_yticklabels(d.index, fontsize=7)
        ax.invert_yaxis()
        ax.set(xlabel="Δ subject accuracy vs full pipeline (ITT)",
               title=f"{sim}, ε={MAIN_EFFECT if sim == 'v2' else ORIGINAL_EFFECT}")
    fig.tight_layout()
    savefig(fig, "E4_ablation")
    plt.show()

## E5. Confound and leakage tests (review 4C, 5C, 5E)
1. **Can nuisance variables be predicted from the features?** A random forest is trained on subject-mean
   features to predict each nuisance variable (subject 5-fold CV). Results near chance support the claim
   that the features do not encode gain, units or sampling rate. Montage type is expected to be predictable
   from the missingness pattern, and age, blinks and EMG from real spectral changes. These are reported
   as such. Note that in **v1** the "gain" scales only the oscillations, not the 1/f background, so it
   changes the spectrum and is expected to be predictable. This is an artefact of the original simulator.
2. **Does P(dyslexic) track any nuisance variable within each class?** Spearman correlations are computed
   with Benjamini–Hochberg correction. ε_i (the per-subject effect size) is a positive control.
3. **Fig. 4 question.** Does the mean P(dyslexic) of controls differ by montage type? This is tested with
   Kruskal–Wallis within each class.

In [ ]:
def bh(p):
    p = np.asarray(p, float)
    o = np.argsort(p)
    q = np.empty_like(p)
    q[o] = np.minimum.accumulate((p[o] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
    return np.minimum(q, 1)


if RUN["E5"]:
    pred_rows, corr_rows, fig4_rows = [], [], []
    targets_c = {"label (reference)": "label", "montage type": "kind", "sampling rate": "fs", "units (V/uV)": "unit",
                 "mains present": "mains_present", "duration": "dur", "blink present": "blink",
                 "EMG present": "emg", "movement transient": "transient", "bad channel injected": "bad_present"}
    targets_r = {"log gain": "log_gain", "age": "age", "alpha peak frequency": "alpha_pf", "1/f exponent": "chi"}
    for sim, eps in [("v2", MAIN_EFFECT), ("v1", ORIGINAL_EFFECT)]:
        for seed in SEEDS_SWEEP:
            coh = build_cohort(seed, sim, eps, PLAN_REF)
            meta = coh["meta"].copy()
            meta["mains_present"] = meta["mains"].notna()
            meta["bad_present"] = meta["bad_type"].notna()
            meta["log_gain"] = np.log(meta["gain"])
            Xs = np.vstack([np.nanmean(coh["feats"][VAR][s], axis=0) if coh["feats"][VAR][s] is not None
                            else np.full(77, np.nan) for s in meta.index])
            ok = np.isfinite(Xs).any(axis=1)
            Xs, meta = Xs[ok], meta[ok]
            kf = StratifiedKFold(5, shuffle=True, random_state=seed)
            for nm, col in targets_c.items():
                yv = np.unique(meta[col].astype(str).to_numpy(), return_inverse=True)[1]   # integer codes
                if len(np.unique(yv)) < 2:
                    continue
                pr = np.empty(len(yv), dtype=int)
                for tr, te in kf.split(Xs, yv):
                    m = Pipeline([("i", SimpleImputer(strategy="median", keep_empty_features=True)),
                                  ("c", RandomForestClassifier(300, min_samples_leaf=2, class_weight="balanced",
                                                               n_jobs=N_JOBS_MODEL, random_state=seed))])
                    pr[te] = m.fit(Xs[tr], yv[tr]).predict(Xs[te])
                pred_rows.append(dict(sim=sim, seed=seed, target=nm, metric="balanced acc",
                                      score=balanced_accuracy_score(yv, pr), chance=1 / len(np.unique(yv))))
            # montage from the missingness pattern alone
            Mz = np.isnan(Xs).astype(float)
            yv = np.unique(meta["kind"].to_numpy(), return_inverse=True)[1]
            pr = np.empty(len(yv), dtype=int)
            for tr, te in kf.split(Mz, yv):
                pr[te] = RandomForestClassifier(100, random_state=seed).fit(Mz[tr], yv[tr]).predict(Mz[te])
            pred_rows.append(dict(sim=sim, seed=seed, target="montage type (missingness pattern only)",
                                  metric="balanced acc", score=balanced_accuracy_score(yv, pr), chance=1 / 3))
            kfr = KFold(5, shuffle=True, random_state=seed)
            for nm, col in targets_r.items():
                yv = meta[col].to_numpy(float)
                if not np.isfinite(yv).all() or yv.std() == 0:
                    continue
                pr = np.empty(len(yv))
                for tr, te in kfr.split(Xs):
                    m = Pipeline([("i", SimpleImputer(strategy="median", keep_empty_features=True)),
                                  ("r", RandomForestRegressor(300, min_samples_leaf=2, n_jobs=N_JOBS_MODEL,
                                                              random_state=seed))])
                    pr[te] = m.fit(Xs[tr], yv[tr]).predict(Xs[te])
                pred_rows.append(dict(sim=sim, seed=seed, target=nm, metric="R2", score=r2_score(yv, pr), chance=0.0))
        # within-class correlations and the Fig. 4 check, pooled over seeds (OOF probabilities)
        D = pd.concat([get_oof(sim, eps, s).assign(seed=s) for s in SEEDS_SWEEP])
        D["log_gain"] = np.log(D["gain"])
        D["unit_V"] = (D["unit"] == "V").astype(int)
        D["mains_present"] = D["mains"].notna().astype(int)
        D["bad_present"] = D["bad_type"].notna().astype(int)
        nuis = ["log_gain", "fs", "dur", "nch", "n_regions", "unit_V", "mains_present", "transient", "bad_present",
                "age", "alpha_pf", "chi", "blink", "emg", "n_epochs"]
        for lab, nmc in [(0, "Control"), (1, "Dyslexic")]:
            d = D[(D.label == lab) & D.p.notna()]
            vs = nuis + (["eps_i"] if lab == 1 else [])
            res = []
            for v in vs:
                x = d[v].astype(float)
                if x.nunique() < 2:
                    continue
                rho, pv = stats.spearmanr(x, d["p"])
                res.append((v, rho, pv))
            q = bh([r_[2] for r_ in res])
            for (v, rho, pv), qq in zip(res, q):
                corr_rows.append(dict(sim=sim, cls=nmc, variable=v, rho=rho, p=pv, q_BH=qq, n=len(d)))
            groups = [d[d.kind == k]["p"].to_numpy() for k in KINDS]
            kw = stats.kruskal(*groups) if all(len(g_) > 1 for g_ in groups) else None
            fig4_rows.append(dict(sim=sim, cls=nmc, **{f"meanP_{k}": np.mean(g_) for k, g_ in zip(KINDS, groups)},
                                  **{f"sdP_{k}": np.std(g_) for k, g_ in zip(KINDS, groups)},
                                  kruskal_p=kw.pvalue if kw else np.nan))
    e5a = pd.DataFrame(pred_rows).groupby(["sim", "target", "metric"]).agg(score=("score", msd), chance=("chance", "first"))
    e5b = pd.DataFrame(corr_rows)
    e5c = pd.DataFrame(fig4_rows)
    e5a.to_csv(RES / "E5_nuisance_predictability.csv")
    e5b.to_csv(RES / "E5_within_class_correlations.csv", index=False)
    e5c.to_csv(RES / "E5_probability_by_montage.csv", index=False)
    log("E5a Can nuisance variables be predicted from the features? (subject-level 5-fold CV; mean ± SD over seeds)", e5a)
    log("E5b Within-class Spearman correlation between OOF P(dyslexic) and nuisance variables (BH-corrected q)",
        e5b.round(4))
    log("E5c Mean OOF P(dyslexic) by montage within each class (Fig. 4 question) + Kruskal-Wallis p", e5c.round(4))

## E7. Electrode-placement and region-coverage study (review 5C, future work)
Test subjects are simulated with all 19 channels and then re-analysed under many channel subsets: random
subsets of k channels (5 draws per subject and k), named layouts, and the full montage with one region
removed. Because the **same subjects** appear under every montage, the differences isolate the effect of
electrode placement. The model is trained on the mixed-montage E2 cohort of the same seed.

In [ ]:
REGION_CH = {r: [c for c in BASE_CANON if eeg.channel_region(c) == r] for r in eeg.REGIONS}
NAMED = {
    "full 19": list(BASE_CANON),
    "OpenBCI 8 (Fp1 Fp2 C3 C4 P7 P8 O1 O2)": list(OPENBCI8_CANON),
    "posterior only (P, O, midline)": [c for c in BASE_CANON if eeg.channel_region(c)[0] in "PO" or c == "PZ"],
    "fronto-central only": [c for c in BASE_CANON if eeg.channel_region(c)[0] in "FC" or c in ("FZ", "CZ")],
    "left hemisphere + midline": [c for c in BASE_CANON if eeg.channel_region(c).endswith("_L") or c in ("FZ", "CZ", "PZ")],
    "right hemisphere + midline": [c for c in BASE_CANON if eeg.channel_region(c).endswith("_R") or c in ("FZ", "CZ", "PZ")],
    "full minus left posterior (T7 P7 P3 O1)": [c for c in BASE_CANON if c not in ("T7", "P7", "P3", "O1")],
}
K_GRID = [4, 6, 8, 10, 12, 14, 16]

if RUN["E7"]:
    rows = []
    for seed in SEEDS_SMALL:
        train = build_cohort(seed, "v2", MAIN_EFFECT, PLAN_REF)
        Xtr, ytr, _ = stack(train, VAR)
        model = make_model("RF", seed).fit(Xtr, ytr)
        r = np.random.default_rng([seed, 77])
        labels = r.permutation([0] * (N_TEST_E7 // 2) + [1] * (N_TEST_E7 - N_TEST_E7 // 2))
        tasks = []
        for i in range(N_TEST_E7):
            mont = [(nm, ch) for nm, ch in NAMED.items()]
            mont += [(f"drop {reg}", [c for c in BASE_CANON if c not in REGION_CH[reg]]) for reg in eeg.REGIONS]
            for k in K_GRID:
                for d_ in range(5 if not QUICK else 2):
                    mont.append((f"random k={k}", [BASE_CANON[j] for j in sorted(r.choice(19, k, replace=False))]))
            tasks.append(dict(sid=300000 + i, label=int(labels[i]), fs=int(r.choice(FS_CHOICES)),
                              dur=int(r.choice(DURATION_CHOICES_S)), effect=MAIN_EFFECT, montages=mont, p=SIM2,
                              ss=np.random.SeedSequence([seed, 2000, i]).entropy))
        res = Parallel(n_jobs=N_JOBS)(delayed(rs._multi_montage_worker)(t) for t in tasks)
        for subj in res:
            for o in subj:
                p = float(model.predict_proba(o["x"])[:, 1].mean()) if o["x"] is not None else np.nan
                rows.append(dict(seed=seed, sid=o["sid"], label=o["label"], montage=o["montage"], n_ch=o["n_ch"],
                                 n_regions=o["n_regions"], p=p))
    e7 = pd.DataFrame(rows)
    e7.to_csv(RES / "E7_electrode_placement_raw.csv", index=False)

    def acc_tab(d):
        m = metrics(d["label"], d["p"])
        return pd.Series(dict(n=len(d), coverage=m["coverage"], acc_itt=m["acc"],
                              acc_processed=metrics(d["label"], d["p"], itt=False)["acc"] if d.p.notna().any() else np.nan,
                              auc=m.get("auc", np.nan)))
    named = e7[~e7.montage.str.startswith(("random", "drop"))].groupby("montage").apply(acc_tab)
    drop = e7[e7.montage.str.startswith("drop") | (e7.montage == "full 19")].groupby("montage").apply(acc_tab)
    drop["d_acc_vs_full"] = drop["acc_processed"] - drop.loc["full 19", "acc_processed"]
    rnd = e7[e7.montage.str.startswith("random")]
    by_k = rnd.groupby("n_ch").apply(acc_tab)
    by_reg = rnd[rnd.p.notna()].groupby("n_regions").apply(acc_tab)
    for t_, nm in [(named, "named"), (drop, "drop_region"), (by_k, "by_k"), (by_reg, "by_regions")]:
        t_.to_csv(RES / f"E7_{nm}.csv")
    log("E7a Named layouts (same test subjects; ITT = refused montages count as errors)", named.round(3))
    log("E7b Full montage with one region removed", drop.round(3))
    log("E7c Random k-channel subsets: accuracy vs number of channels", by_k.round(3))
    log("E7d Random subsets (processed only): accuracy vs number of regions covered", by_reg.round(3))
    # partial association: regions vs channels as predictors of correctness (logistic regression)
    d = rnd[rnd.p.notna()].copy()
    d["correct"] = ((d.p > 0.5) == d.label).astype(int)
    from sklearn.linear_model import LogisticRegression as _LR
    Z = StandardScaler().fit_transform(d[["n_ch", "n_regions"]])
    coef = _LR().fit(Z, d["correct"]).coef_[0]
    log("E7e Logistic regression of correctness on standardised (n_channels, n_regions) - random subsets",
        f"coef n_channels = {coef[0]:+.3f}, coef n_regions = {coef[1]:+.3f}, "
        f"Spearman(n_ch, n_regions) = {stats.spearmanr(d.n_ch, d.n_regions)[0]:.2f}")
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
    ax[0].plot(by_k.index, by_k["acc_processed"], "o-", color=C_DYS, label="processed")
    ax[0].plot(by_k.index, by_k["acc_itt"], "s--", color="0.5", label="ITT (refusals = errors)")
    ax[0].set(xlabel="number of channels (random subsets)", ylabel="subject accuracy", ylim=(0.4, 1))
    ax[0].legend(frameon=False, fontsize=7)
    ax[1].plot(by_reg.index, by_reg["acc_processed"], "o-", color=C_DYS)
    ax[1].set(xlabel="number of regions covered", ylabel="subject accuracy", ylim=(0.4, 1))
    dd = drop.drop("full 19").sort_values("d_acc_vs_full")
    ax[2].barh(range(len(dd)), dd["d_acc_vs_full"], color=C_DYS)
    ax[2].set_yticks(range(len(dd)))
    ax[2].set_yticklabels([s.replace("drop ", "") for s in dd.index], fontsize=7)
    ax[2].axvline(0, color="k", lw=0.7)
    ax[2].set(xlabel="Δ accuracy when region removed")
    fig.tight_layout()
    savefig(fig, "E7_electrode_placement")
    plt.show()

## E8. Classifier comparison with valid statistics (review: abstract, 5D)
* **Primary test.** Every seed is an independent cohort, so the per-seed differences between two models
  are independent. A paired t-test and a Wilcoxon test across seeds are therefore valid. Holm correction is
  applied, and a Friedman test is run over all models.
* **Nadeau–Bengio corrected resampled t-test.** Five repeats of 5-fold CV on one cohort, with the
  variance corrected for overlapping training sets.
* **DeLong test.** Subject-level AUC on the 75/25 hold-out of one cohort.
* **Timing.** Mean fit time per fold, together with the hardware string.

In [ ]:
MODELS = ["RF", "GB", "LR", "SVM", "LDA", "kNN", "HGB"]
MODEL_NAMES = {"RF": "Random forest", "GB": "Gradient boosting", "LR": "Logistic regression", "SVM": "SVM (RBF)",
               "LDA": "LDA", "kNN": "kNN (k=15)", "HGB": "HistGradientBoosting (native NaN)"}


def delong(y, p1, p2):
    """DeLong test for two correlated ROC AUCs (Sun & Xu fast algorithm). Returns (auc1, auc2, p)."""
    y = np.asarray(y)
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    m, n = len(pos), len(neg)

    def comps(p):
        v10 = np.array([(np.sum(p[i] > p[neg]) + 0.5 * np.sum(p[i] == p[neg])) / n for i in pos])
        v01 = np.array([(np.sum(p[pos] > p[j]) + 0.5 * np.sum(p[pos] == p[j])) / m for j in neg])
        return v10.mean(), v10, v01
    a1, v10a, v01a = comps(np.asarray(p1))
    a2, v10b, v01b = comps(np.asarray(p2))
    s10 = np.cov(np.vstack([v10a, v10b]))
    s01 = np.cov(np.vstack([v01a, v01b]))
    var = (s10[0, 0] + s10[1, 1] - 2 * s10[0, 1]) / m + (s01[0, 0] + s01[1, 1] - 2 * s01[0, 1]) / n
    z = (a1 - a2) / np.sqrt(var) if var > 0 else 0.0
    return a1, a2, 2 * stats.norm.sf(abs(z))


def holm(p):
    p = np.asarray(p, float)
    o = np.argsort(p)
    adj = np.empty_like(p)
    adj[o] = np.minimum(1, np.maximum.accumulate(p[o] * (len(p) - np.arange(len(p)))))
    return adj


if RUN["E8"]:
    rows = []
    E8_CONFIGS = [("v2", MAIN_EFFECT, SEEDS_MAIN), ("v1", ORIGINAL_EFFECT, SEEDS_SWEEP)]
    for sim, eps, seeds in E8_CONFIGS:
        for seed in seeds:
            coh = build_cohort(seed, sim, eps, PLAN_REF)
            meta = coh["meta"]
            folds = subject_folds(meta, seed)
            for mdl in MODELS:
                p, times = oof_subject_probs(coh, VAR, folds, model=mdl, seed=seed, return_times=True)
                m = metrics(meta["label"], p)
                rows.append(dict(sim=sim, eps=eps, seed=seed, model=mdl, fit_s=np.mean(times), **m))
    e8 = pd.DataFrame(rows)
    e8.to_csv(RES / "E8_classifiers_raw.csv", index=False)
    tabs = []
    for (sim, eps), d in e8.groupby(["sim", "eps"]):
        piv_auc = d.pivot(index="seed", columns="model", values="auc")
        piv_acc = d.pivot(index="seed", columns="model", values="acc")
        fr = stats.friedmanchisquare(*[piv_auc[c] for c in MODELS]).pvalue if len(piv_auc) > 2 else np.nan
        t_p = [stats.ttest_rel(piv_auc[c], piv_auc["RF"]).pvalue if c != "RF" else np.nan for c in MODELS]
        w_p = [stats.wilcoxon(piv_auc[c], piv_auc["RF"]).pvalue if (c != "RF" and len(piv_auc) > 5) else np.nan
               for c in MODELS]
        hp = np.full(len(MODELS), np.nan)
        idx = [i for i, c in enumerate(MODELS) if c != "RF"]
        hp[idx] = holm(np.array(t_p)[idx])
        tab = pd.DataFrame(dict(sim=sim, eps=eps, model=[MODEL_NAMES[c] for c in MODELS],
                                auc=[msd(piv_auc[c]) for c in MODELS], acc=[msd(piv_acc[c]) for c in MODELS],
                                d_auc_vs_RF=[msd(piv_auc[c] - piv_auc["RF"]) for c in MODELS],
                                p_t_vs_RF=t_p, p_holm=hp, p_wilcoxon=w_p,
                                fit_s_per_fold=[d[d.model == c]["fit_s"].mean() for c in MODELS],
                                friedman_p=fr))
        tabs.append(tab)
    e8_tab = pd.concat(tabs)
    e8_tab.to_csv(RES / "E8_classifiers.csv", index=False)
    log(f"E8a Classifiers: subject-level 5-fold CV across independent seeds; paired tests vs RF. Hardware: {HARDWARE}",
        e8_tab.round(4))

    # Nadeau-Bengio corrected resampled t-test (5x5 CV on one cohort) and DeLong on the hold-out
    seed = SEEDS_MAIN[0]
    coh = build_cohort(seed, "v2", MAIN_EFFECT, PLAN_REF)
    meta = coh["meta"]
    R = 2 if QUICK else 5
    fold_auc = {mdl: [] for mdl in MODELS}
    for rep in range(R):
        folds = subject_folds(meta, 1000 + rep)
        for mdl in MODELS:
            p = oof_subject_probs(coh, VAR, folds, model=mdl, seed=seed)
            for f in range(K_FOLDS):
                s_ = folds.index[folds == f]
                fold_auc[mdl].append(roc_auc_score(meta.loc[s_, "label"], p.loc[s_]))
    J = len(fold_auc["RF"])
    nb_rows = []
    for mdl in MODELS:
        if mdl == "RF":
            continue
        dlt = np.array(fold_auc[mdl]) - np.array(fold_auc["RF"])
        var = dlt.var(ddof=1)
        t = dlt.mean() / np.sqrt((1 / J + (1 / K_FOLDS) / (1 - 1 / K_FOLDS)) * var) if var > 0 else 0.0
        nb_rows.append(dict(model=MODEL_NAMES[mdl], mean_d_auc=dlt.mean(), t=t, p_corrected=2 * stats.t.sf(abs(t), J - 1)))
    sids = meta.index.to_numpy()
    tr, te = next(GroupShuffleSplit(1, test_size=0.25, random_state=seed).split(sids, groups=sids))
    ph = {mdl: fit_predict(coh, sids[tr], coh, sids[te], model=mdl, seed=seed) for mdl in MODELS}
    yte = meta.loc[sids[te], "label"].to_numpy()
    for row in nb_rows:
        mdl = [k for k, v in MODEL_NAMES.items() if v == row["model"]][0]
        a1, a2, pdl = delong(yte, ph[mdl].to_numpy(), ph["RF"].to_numpy())
        row.update(holdout_auc=a1, holdout_auc_RF=a2, p_delong=pdl)
    e8b = pd.DataFrame(nb_rows)
    e8b["p_corrected_holm"] = holm(e8b["p_corrected"])
    e8b["p_delong_holm"] = holm(e8b["p_delong"])
    e8b.to_csv(RES / "E8_nadeau_bengio_delong.csv", index=False)
    log(f"E8b v2 ε={MAIN_EFFECT}, seed {seed}: Nadeau-Bengio corrected t-test ({R}x{K_FOLDS} CV, subject AUC per fold) "
        "and DeLong test on the 75/25 hold-out, each model vs RF", e8b.round(4))

## E9. Permutation importance on held-out data (review 4C, Fig. 5)
Per-feature permutation importance is the drop in held-out epoch AUC. Feature *families* are permuted
jointly: by band, and by location. Readable labels replace the code names.

In [ ]:
REG_LABEL = {"F_L": "L frontal", "F_R": "R frontal", "C_L": "L central", "C_R": "R central", "T_L": "L temporal",
             "T_R": "R temporal", "P_L": "L parietal", "P_R": "R parietal", "O_L": "L occipital",
             "O_R": "R occipital", "MID": "Midline", "global": "Global"}
BAND_LABEL = {"delta": "δ", "theta": "θ", "alpha": "α", "beta": "β", "gamma": "γ", "logTBR": "log θ/β"}
LOBE_LABEL = {"F": "frontal", "C": "central", "T": "temporal", "P": "parietal", "O": "occipital"}


def pretty(name):
    if name.startswith("asym_alpha_"):
        return f"α asymmetry ({LOBE_LABEL[name[-1]]})"
    loc, band = name.rsplit("_", 1)
    return f"{REG_LABEL[loc]} {BAND_LABEL[band]}"


FN = eeg.FEATURE_NAMES
FAMILIES = {**{f"band: {BAND_LABEL[b]}": [i for i, n in enumerate(FN) if n.endswith("_" + b)] for b in BAND_LABEL},
            "band: α asymmetry": [i for i, n in enumerate(FN) if n.startswith("asym")],
            **{f"location: {REG_LABEL[r]}": [i for i, n in enumerate(FN) if n.startswith(r + "_")] for r in REG_LABEL}}

if RUN["E9"]:
    imp_rows, fam_rows = [], []
    for sim, eps in [("v2", MAIN_EFFECT), ("v1", ORIGINAL_EFFECT)]:
        for seed in SEEDS_SWEEP:
            coh = build_cohort(seed, sim, eps, PLAN_REF)
            sids = coh["meta"].index.to_numpy()
            tr, te = next(GroupShuffleSplit(1, test_size=0.25, random_state=seed).split(sids, groups=sids))
            Xtr, ytr, _ = stack(coh, VAR, sids[tr])
            Xte, yte, _ = stack(coh, VAR, sids[te])
            model = make_model("RF", seed).fit(Xtr, ytr)
            base = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
            pi = permutation_importance(model, Xte, yte, scoring="roc_auc", n_repeats=3 if QUICK else 10,
                                        random_state=seed, n_jobs=1)
            gini = model.named_steps["clf"].feature_importances_
            for i, n in enumerate(FN):
                imp_rows.append(dict(sim=sim, seed=seed, feature=n, label=pretty(n), perm=pi.importances_mean[i], gini=gini[i]))
            r = np.random.default_rng(seed)
            for fam, cols in FAMILIES.items():
                drops = []
                for _ in range(3 if QUICK else 10):
                    Xp = Xte.copy()
                    Xp[:, cols] = Xte[r.permutation(len(Xte))][:, cols]
                    drops.append(base - roc_auc_score(yte, model.predict_proba(Xp)[:, 1]))
                fam_rows.append(dict(sim=sim, seed=seed, family=fam, n_features=len(cols), auc_drop=np.mean(drops)))
    e9 = pd.DataFrame(imp_rows)
    e9.to_csv(RES / "E9_permutation_importance_raw.csv", index=False)
    e9_tab = e9.groupby(["sim", "label"]).agg(perm_mean=("perm", "mean"), perm_sd=("perm", "std"),
                                               gini_mean=("gini", "mean")).reset_index()
    e9_tab["rank_perm"] = e9_tab.groupby("sim")["perm_mean"].rank(ascending=False)
    e9_tab["rank_gini"] = e9_tab.groupby("sim")["gini_mean"].rank(ascending=False)
    top = e9_tab.sort_values(["sim", "perm_mean"], ascending=[True, False]).groupby("sim").head(15)
    top.to_csv(RES / "E9_top15.csv", index=False)
    log("E9a Top-15 features by held-out permutation importance (Δ epoch AUC; mean over seeds) with Gini rank", top.round(4))
    e9f = pd.DataFrame(fam_rows).groupby(["sim", "family"]).agg(n=("n_features", "first"), auc_drop=("auc_drop", msd))
    e9f.to_csv(RES / "E9_family_importance.csv")
    log("E9b Grouped permutation importance by feature family (Δ held-out epoch AUC)", e9f)
    for sim in ["v2", "v1"]:
        t15 = top[top.sim == sim].iloc[::-1]
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.barh(t15["label"], t15["perm_mean"], xerr=t15["perm_sd"], color=C_ACC, error_kw=dict(lw=0.7))
        ax.set(xlabel="Permutation importance (Δ held-out AUC)",
               title=f"Top 15 features - {sim}, ε={MAIN_EFFECT if sim == 'v2' else ORIGINAL_EFFECT}")
        fig.tight_layout()
        savefig(fig, f"E9_permutation_importance_{sim}")
        plt.show()

## E12. Sensitivity of the EGI site-distance threshold (review 4B)
The first table maps the EGI-128 net (no simulation needed): electrodes kept, distinct 10-10 sites and regions.
The second gives the accuracy on unseen EGI cohorts processed with each threshold.

In [ ]:
THRESHOLDS = [0.25, 0.30, 0.34, 0.38, 0.42, 0.46, 0.50]
if RUN["E12"] and HAVE_MNE:
    rows = []
    for thr in THRESHOLDS:
        eeg._MAX_SITE_DISTANCE = thr
        res = eeg._position_route(EGI_NAMES, None, 1)
        keep, canon = (res[0], res[1]) if res else ([], [])
        rows.append(dict(threshold=thr, electrodes_kept=len(keep), distinct_sites=len(set(canon)),
                         regions=n_regions(canon) if canon else 0,
                         per_region=dict(pd.Series([eeg.channel_region(c) for c in canon]).value_counts()) if canon else {}))
    eeg._MAX_SITE_DISTANCE = rs.DEFAULT_SITE_DISTANCE
    e12a = pd.DataFrame(rows)
    acc_rows = []
    for seed in SEEDS_SMALL:
        train = build_cohort(seed, "v2", MAIN_EFFECT, PLAN_REF)
        Xtr, ytr, _ = stack(train, VAR)
        model = make_model("RF", seed).fit(Xtr, ytr)
        for thr in THRESHOLDS:
            egi = build_cohort(seed + 600, "v2", MAIN_EFFECT, PLAN_REF, n=N_EGI, net="egi", sid_offset=100000,
                               site_dist=thr, tag="egithr_")
            Xte, yte, gte = stack(egi, VAR)
            ps = pd.Series(np.nan, index=egi["meta"].index)
            if len(Xte):
                sp = subject_mean(model.predict_proba(Xte)[:, 1], gte)
                ps.loc[sp.index] = sp.to_numpy()
            m = metrics(egi["meta"]["label"], ps)
            acc_rows.append(dict(seed=seed, threshold=thr, acc=m["acc"], auc=m.get("auc", np.nan), coverage=m["coverage"]))
    e12b = pd.DataFrame(acc_rows).groupby("threshold").agg(acc=("acc", msd), auc=("auc", msd), coverage=("coverage", "mean"))
    e12 = e12a.set_index("threshold").join(e12b)
    e12.to_csv(RES / "E12_egi_threshold.csv")
    log("E12 EGI-128 site-distance threshold: mapping and accuracy on unseen EGI cohorts", e12)

## E13. Recording length (review 4A)
The model is trained on the standard 24–40 s cohort and tested on new subjects recorded for longer.

In [ ]:
DURS = [24, 60] if QUICK else [24, 60, 120, 180]
if RUN["E13"]:
    rows = []
    for seed in SEEDS_SMALL:
        train = build_cohort(seed, "v2", MAIN_EFFECT, PLAN_REF)
        for dur in DURS:
            test = build_cohort(seed + 3000, "v2", MAIN_EFFECT, PLAN_REF, n=N_TEST_E7, durations=[dur],
                                sid_offset=200000, tag=f"dur{dur}_")
            p = fit_predict(train, train["meta"].index, test, test["meta"].index, seed=seed)
            m = metrics(test["meta"]["label"], p)
            rows.append(dict(seed=seed, duration_s=dur, epochs_per_subject=test["meta"]["n_epochs"].mean(), **m))
    e13 = pd.DataFrame(rows).groupby("duration_s").agg(epochs=("epochs_per_subject", "mean"), acc=("acc", msd),
                                                        auc=("auc", msd), sens90=("sens90", msd))
    e13.to_csv(RES / "E13_recording_length.csv")
    log("E13 Accuracy vs recording length (v2, model trained on 24-40 s recordings)", e13)

## Summary file and archive
**Send me `revision_results/results_summary.txt`.** The zip also contains every CSV and figure, but not the cache.

In [ ]:
(RES / "results_summary.txt").write_text("\n".join(SUMMARY), encoding="utf-8")
zpath = WORK_DIR / "revision_results.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in RES.rglob("*"):
        if f.is_file() and CACHE not in f.parents:
            zf.write(f, f.relative_to(WORK_DIR))
for f in [WORK_DIR / "revision_sim.py", MODULE_PATH]:
    with zipfile.ZipFile(zpath, "a") as zf:
        zf.write(f, f.name)
print(f"wrote {RES / 'results_summary.txt'} ({len(SUMMARY)} blocks) and {zpath}")